# Chapter 6 Systems of Linear Equations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asj252/linear-algebra-with-python/blob/main/en/EN_Chap6_Linear_System.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/asj252/linear-algebra-with-python/HEAD?labpath=en%2FEN_Chap6_Linear_System.ipynb)

The word *fangcheng*, the modern Chinese term for "equation," is older, and carries more meaning, than most people realize.

Chapter 8 of *The Nine Chapters on the Mathematical Art* is titled *Fangcheng* ("Rectangular Arrays"); the book was compiled and edited over a long period of accumulation. Its method lays out counting rods as an array of numbers: the coefficients of each equation are arranged along a vertical column, the unknowns are eliminated step by step by multiplying and subtracting, and the solution is then found by back substitution. This is an important record of the ancient idea of elimination; what was used at the time was counting rods on a counting board, not the modern abacus.

In modern notation, writing each equation horizontally as a row gives the **augmented matrix**, and the corresponding elimination steps are written as **row operations**. The ancient arrays and modern matrices follow different conventions of orientation, but both use equivalence transformations that leave the solutions unchanged. This connection helps us understand the history, but it does not mean that the ancients already used the modern concept of a matrix.

By the middle of the nineteenth century, systems of linear equations had begun to play an increasingly important role in problems of physics and engineering.

In 1845, Gustav Kirchhoff studied the relations between currents and voltages in electrical networks. Conservation of current at the nodes and the voltage relations around loops, together with the constitutive equations of the components, provided a systematic framework for circuit analysis. For a linear resistive network these relations form a system of linear equations; the number of independent equations must take the topological constraints of the network into account, and is not obtained by simply adding up the numbers of nodes, branches, or loops.

**Under a suitable linear model, the connection and equilibrium conditions of a physical system can be translated into a system of linear equations.** Nonlinear components, by contrast, usually require nonlinear solves and local linearization.

This framework extends to modern circuit simulation: many algorithms solve linear systems over and over in the course of their iterations. Multi-antenna channel equalization in communications also often involves linear equations and least-squares problems. Different applications rest on different modeling assumptions, yet they share one body of linear algebra tools.

More profoundly, solving a system of equations means far more than arithmetic elimination. From the modern geometric viewpoint, solving $\mathbf{A}\mathbf{x} = \mathbf{b}$ is at once a search for the common intersection point of several hyperplanes in a high-dimensional space and a test of whether the target vector $\mathbf{b}$ can be formed as a linear combination of the column vectors of the matrix.

The grain calculations on ancient counting boards, the resistive networks in the notes of nineteenth-century physicists, and today's simulations of electromagnetic fields on supercomputers can all, under suitable models, be reduced to systems of linear equations. This consistency across civilizations, engineering, and pure geometry is an important reason why systems of linear equations sit at the heart of linear algebra.

### Chapter Structure and Learning Objectives

The elimination algorithm of *The Nine Chapters* is intuitive and effective, but what structure lies behind it? Why can those operations be applied freely without changing the solutions of the system? §6.1 starts from this question and interprets row operations as invertible linear transformations—left multiplication by elementary matrices—thereby raising the arithmetic act of "solving equations" to the language of linear algebra.

Each of these elementary operations corresponds to an invertible elementary matrix, so they preserve the dimensions of the row space and the column space, which guarantees the legitimacy of every subsequent reduction step in the chapter.

§6.2 introduces the row echelon form and the reduced row echelon form, the end point of elimination: once a matrix has been brought to this canonical form, the structure of the solutions—no solution, a unique solution, or infinitely many solutions—becomes clear at a glance. Here the concept of rank receives its most concrete computational definition, and it formally connects with the concept of dimension from Chapter 4 on abstract linear spaces.

§6.3 contains the central algorithms of the chapter: Gaussian elimination and the LU decomposition. The former is the procedure for solving; the latter is the same procedure in the language of matrices—factoring $\mathbf{A}$ as the product of a lower triangular matrix $\mathbf{L}$ and an upper triangular matrix $\mathbf{U}$. The significance of the LU decomposition goes beyond computational efficiency: it records every step of the elimination permanently in $\mathbf{L}$, so that the same coefficient matrix can be reused for different right-hand-side vectors, which is one of the basic design principles of modern numerical linear algebra.

§6.4 closes with three concrete applications: the balance equations of network flows, the conservation of atoms in chemical reactions, and the static equilibrium of mechanical systems. These three problems come from different fields, yet mathematically they are all $\mathbf{Ax} = \mathbf{b}$—they share one theory and one set of algorithms.

Once you have read this chapter, you will see a different depth of field in the ancient name of the *fangcheng* method.


## 6.1 Elementary Matrix Operations

We begin with the most basic matrix operations and set out to explore the mysteries of systems of linear equations.

Elementary matrix operations are the basic tools for solving systems of linear equations. In essence they are an algebraic upgrade of the three operations for solving equations that you already know from high school—multiplying by a constant, interchanging two equations, and adding two equations. Let us first look at a concrete example to get a feel for this upgrade:

Multiplying row 0 of the system by 2 is completely equivalent to "left-multiplying by the diagonal matrix $\mathbf{E}_0(2)$":

$$
\underbrace{\begin{bmatrix} 2 & 0 \\ 0 & 1 \end{bmatrix}}_{\mathbf{E}_0(2)}
\underbrace{\begin{bmatrix} 1 & 3 \\ 2 & 5 \end{bmatrix}}_{\mathbf{A}}
= \begin{bmatrix} 2 & 6 \\ 2 & 5 \end{bmatrix}
= \underbrace{\begin{bmatrix} 2 \cdot \text{row}_0(\mathbf{A}) \\ \text{row}_1(\mathbf{A}) \end{bmatrix}}_{\text{result of row scaling}}
$$

This observation reveals the central theme of this section: every elimination operation done by hand corresponds to left multiplication by a particular elementary matrix. Once you master this correspondence, Gaussian elimination turns from a "technique" into a "structure."

```{important}
:label: imp-matrix-ops
Elementary matrix operations can be expressed as left or right multiplication of the matrix $\mathbf{A}$ by an elementary matrix:
- Left multiplication by an elementary matrix, $\mathbf{EA}$, corresponds to a **row operation**: it applies an invertible linear transformation on the output side, and the domain and codomain themselves are unchanged
- Right multiplication by an elementary matrix, $\mathbf{AE}$, corresponds to a **column operation**: it applies an invertible linear transformation on the input side, and the domain and codomain themselves are unchanged
- All elementary operations preserve the **dimensions of the row space and the column space** of the matrix, and therefore also preserve its rank
```

### 6.1.1 Row Scaling

**Row scaling** means multiplying one row of a matrix by a nonzero constant.

:::{prf:definition} Row Scaling Operation
:label: def-row-scaling
For row $i$ of the matrix $\mathbf{A}$, $\text{row}_i(\mathbf{A})$, the row scaling operation is defined as $\text{row}_i(\mathbf{A}) \rightarrow c \cdot \text{row}_i(\mathbf{A})$, where $c \neq 0$.

This is equivalent to left multiplication by a diagonal matrix $\mathbf{E}_i(c)$, where

$$
[\mathbf{E}_i(c)]_{jk} = 
\begin{cases}
c & \text{if } j = k = i \\
1 & \text{if } j = k \neq i \\
0 & \text{if } j \neq k
\end{cases}
$$

That is, $\mathbf{E}_i(c)\mathbf{A}$ multiplies row $i$ of $\mathbf{A}$ by $c$.
:::

Row scaling does not change the dimension of the row space, because it merely stretches one row vector by a nonzero factor and does not change linear dependence. At the same time, the dimension of the column space is also unchanged, because $\mathbf{E}_i(c)$ is invertible and an invertible transformation does not change the dimension of the image. Since the rank is defined as the dimension of the column space, row scaling does not change the rank of the matrix.

In a system of linear equations $\mathbf{Ax=b}$, row scaling amounts to multiplying both sides of one equation by a nonzero constant, which **does not change the solution set of the equation**.

:::{prf:example} Row Scaling of a System in Two Unknowns
:label: ex-row-scaling
Consider the system of linear equations

$$\begin{cases}x_0 + 2x_1 = 3 \\
2x_0 - x_1 = 4\end{cases}$$

In matrix form, $\mathbf{Ax=b}$, where

$$
\mathbf{A} = \begin{bmatrix} 
1 & 2 \\ 
2 & -1 
\end{bmatrix}, \quad
\mathbf{x} = \begin{bmatrix} 
x_0 \\ 
x_1 
\end{bmatrix}, \quad
\mathbf{b} = \begin{bmatrix} 
3 \\ 
4 
\end{bmatrix}
$$

Multiplying equation 0 by 2 amounts to left multiplication by the matrix

$$
\mathbf{E}_0(2) = \begin{bmatrix} 
2 & 0 \\ 
0 & 1 
\end{bmatrix}
$$

which gives

$$
\mathbf{E}_0(2)\mathbf{A} = \begin{bmatrix} 
2 & 4 \\ 
2 & -1 
\end{bmatrix}, \quad
\mathbf{E}_0(2)\mathbf{b} = \begin{bmatrix} 
6 \\ 
4 
\end{bmatrix}
$$

This changes neither the solution $x_0=11/5, x_1=2/5$ of the original system nor the dimension of the column space of $\mathbf{A}$, so the rank stays the same.
:::




### 6.1.2 Row Exchange

**Row exchange** means interchanging the positions of two rows of a matrix.

:::{prf:definition} Row Exchange Operation
:label: def-row-exchange
For row $i$, $\text{row}_i(\mathbf{A})$, and row $j$, $\text{row}_j(\mathbf{A})$, of the matrix $\mathbf{A}$, the row exchange operation is defined as $\text{row}_i(\mathbf{A}) \leftrightarrow \text{row}_j(\mathbf{A})$.

This is equivalent to left multiplication by a permutation matrix $\mathbf{P}_{ij}$, where

$$
[\mathbf{P}_{ij}]_{kl} = 
\begin{cases}
1 & \text{if } (k,l) \in \{(i,j), (j,i)\} \\
1 & \text{if } k = l \text{ and } k ∉ \{i,j\} \\
0 & \text{otherwise}
\end{cases}
$$

That is, $\mathbf{P}_{ij}\mathbf{A}$ interchanges rows $i$ and $j$ of $\mathbf{A}$.
:::

Row exchange does not change the dimension of the row space, because interchanging two rows merely reorders the row vectors without changing what their linear combinations can produce. Since $\mathbf{P}_{ij}$ is an invertible matrix (in fact an orthogonal matrix), the dimension of the column space is also unchanged. Therefore row exchange preserves the rank of the matrix.

The row exchange operation amounts to reordering the equations of a system of linear equations, which **does not change the solution set of the system**.

```{note}
:label: note-row-exchange
The row exchange matrix $\mathbf{P}_{ij}$ is an orthogonal matrix satisfying $\mathbf{P}_{ij}^{\top} = \mathbf{P}_{ij}^{-1} = \mathbf{P}_{ij}$; that is, it is its own inverse. This reflects the invertibility and the symmetry of the exchange operation.
```

### 6.1.3 Row Elimination

**Row elimination** means adding a multiple of one row to another row, with the aim of eliminating the entry in a particular position.

:::{prf:definition} Row Elimination Operation
:label: def-row-elimination
For row $i$, $\text{row}_i(\mathbf{A})$, and row $j$, $\text{row}_j(\mathbf{A})$, of the matrix $\mathbf{A}$, the row elimination operation is defined as $\text{row}_i(\mathbf{A}) \rightarrow \text{row}_i(\mathbf{A}) + c \cdot \text{row}_j(\mathbf{A})$, where $c$ is some constant and $i\neq j$.

This is equivalent to left multiplication by an elementary matrix $\mathbf{E}_{ij}(c)$, where

$$
[\mathbf{E}_{ij}(c)]_{kl} = 
\begin{cases}
c & \text{if } k = i \text{ and } l = j \\
1 & \text{if } k = l \\
0 & \text{otherwise}
\end{cases}
$$

That is, $\mathbf{E}_{ij}(c)\mathbf{A}$ adds $c$ times row $j$ of $\mathbf{A}$ to row $i$.
:::

Row elimination does not change the dimension of the row space, because it merely replaces one row vector by a linear combination of that vector and another row vector, and the span of all the row vectors stays the same. Since $\mathbf{E}_{ij}(c)$ is an invertible matrix (its inverse is $\mathbf{E}_{ij}(-c)$), the dimension of the column space is also unchanged. Therefore the row elimination operation preserves the rank of the matrix.

Row elimination is the central operation for solving systems of linear equations: it simplifies the system by eliminating variables, but it **does not change the solution set of the system**.

```{tip}
:label: tip-row-elimination
The inverse of the row elimination matrix $\mathbf{E}_{ij}(c)$ is simple and clear: $\mathbf{E}_{ij}^{-1}(c) = \mathbf{E}_{ij}(-c)$. That is, performing the same operation with the opposite coefficient undoes the original operation.
```

:::{prf:example} Row Elimination in a System of Two Equations
:label: ex-row-elimination
Consider the system of linear equations
$\begin{cases}x_0 + 2x_1 = 3 \\
2x_0 - x_1 = 4\end{cases}$

Adding $-2$ times equation 0 to equation 1, the corresponding elementary matrix is

$$
\mathbf{E}_{10}(-2) = \begin{bmatrix}
1 & 0 \\
-2 & 1
\end{bmatrix}
$$

The result of the operation is

$$
\mathbf{E}_{10}(-2)\mathbf{A} = \begin{bmatrix}
1 & 2 \\
0 & -5
\end{bmatrix}, \quad
\mathbf{E}_{10}(-2)\mathbf{b} = \begin{bmatrix}
3 \\
-2
\end{bmatrix}
$$

Although the form of the matrix has changed, the dimension of the column space stays the same (still 2), so the rank is unchanged. After simplification we obtain
$\begin{cases}x_0 + 2x_1 = 3 \\
-5x_1 = -2\end{cases}$

Solving gives $x_1 = \frac{2}{5}$, and substituting into equation 0 gives $x_0=\frac{11}{5}$.
:::

### 6.1.4 The Relation Between Row Operations and Column Operations

Column operations and row operations are dual to each other, and the connection between them is made through the matrix transpose. A column operation on the matrix $\mathbf{A}$ is equivalent to the corresponding row operation on its transpose $\mathbf{A}^{\top}$.

:::{prf:property} Transpose Relation Between Column and Row Operations
:label: pro-row-col-duality
For a matrix $\mathbf{A}$ and an elementary matrix $\mathbf{E}$,
1. $(\mathbf{EA})^{\top} = \mathbf{A}^{\top}\mathbf{E}^{\top}$
2. $(\mathbf{AE})^{\top} = \mathbf{E}^{\top}\mathbf{A}^{\top}$

This means that performing a column operation on $\mathbf{A}$ is equivalent to performing the corresponding row operation on $\mathbf{A}^{\top}$.
:::

From the property above and the fact that row operations preserve the dimension of the column space, we can deduce that column operations preserve the dimension of the row space. Since $\dim(\text{Col}(\mathbf{A})) = \dim(\text{Row}(\mathbf{A}^{\top}))$, column operations do not change the dimension of the row space, and row operations do not change the dimension of the column space; hence all elementary operations preserve the rank of the matrix.

**Column scaling**

**Column scaling** means multiplying one column of a matrix by a nonzero constant; it is equivalent to right multiplication by a diagonal elementary matrix $\mathbf{E}_j(c)$, written $\mathbf{A}\mathbf{E}_j(c)$.

Column scaling does not change the dimension of the column space, because it merely stretches one column vector by a nonzero factor and does not change linear dependence. By the definition of the rank of a matrix (the dimension of the column space), column scaling does not change the rank of the matrix.

**Column exchange**

**Column exchange** means interchanging the positions of two columns of a matrix; it is equivalent to right multiplication by a permutation matrix $\mathbf{P}_{ij}$, written $\mathbf{A}\mathbf{P}_{ij}$.

Column exchange does not change the dimension of the column space, because interchanging two columns merely reorders the column vectors without changing what their linear combinations can produce. Therefore column exchange preserves the rank of the matrix.

**Column elimination**

**Column elimination** means adding a multiple of one column to another column; it is equivalent to right multiplication by an elementary matrix $\mathbf{E}_{ji}(c)$, written $\mathbf{A}\mathbf{E}_{ji}(c)$.

Column elimination does not change the dimension of the column space, because it merely replaces one column vector by a linear combination of that vector and another column vector, and the span of all the column vectors stays the same. Therefore the column elimination operation preserves the rank of the matrix.

```{attention} The Key Difference Between Column Operations and Row Operations
:label: att-col-operations
- Row operations apply an invertible linear transformation on the output side (left multiplication by an invertible matrix) and leave the solution set of the system unchanged; the domain and codomain themselves are unchanged
- Column operations may change the solution set of the system, but they preserve the dimension of the column space (that is, the rank)
- Column operations are very useful for finding a basis of the kernel or for matrix decompositions
```

### 6.1.5 Elementary Operations and Subspaces

Every elementary matrix operation preserves the dimensions of the row space and the column space of the matrix; this is the key to understanding the structure of the solutions of a system of linear equations. We give a slightly more general statement.

:::{prf:theorem} Preservation of Subspace Dimensions
:label: thm-subspace-dim-preservation
Let $\mathbb{K}$ be a field. For any matrix $\mathbf{A} \in M_{m \times n}(\mathbb{K})$ and invertible matrices $\mathbf{P} \in M_{m \times m}(\mathbb{K})$ and $\mathbf{Q} \in M_{n \times n}(\mathbb{K})$:
1. $\dim(\text{Col}(\mathbf{PA})) = \dim(\text{Col}(\mathbf{A}))$
2. $\dim(\text{Row}(\mathbf{PA})) = \dim(\text{Row}(\mathbf{A}))$
3. $\dim(\text{Col}(\mathbf{AQ})) = \dim(\text{Col}(\mathbf{A}))$
4. $\dim(\text{Row}(\mathbf{AQ})) = \dim(\text{Row}(\mathbf{A}))$

In particular, the rank of a matrix, $\text{rank}(\mathbf{A}) = \dim(\text{Col}(\mathbf{A}))$, is unchanged under any invertible matrix operation.
:::

:::{prf:proof} Proof of the Preservation of Subspace Dimensions
:label: prf-subspace-dim-preservation

**Proof of (1): row operations preserve the dimension of the column space**

1. Consider the linear transformation $T_{\mathbf{P}}: \mathbb{K}^m \rightarrow \mathbb{K}^m$ corresponding to the invertible matrix $\mathbf{P}$

2. By the result of Chapter 4 that invertible matrices preserve the dimensions of subspaces ({prf:ref}`cor-inversible-preserve-subspace`), for any subspace $U \subseteq \mathbb{K}^m$:


   $$\dim(\mathbf{P}(U)) = \dim(U)$$

3. Applying this to $U = \text{Col}(\mathbf{A})$ gives
 
   $$\underline{\dim(\text{Col}(\mathbf{PA})) = \dim(\mathbf{P}(\text{Col}(\mathbf{A}))) = \dim(\text{Col}(\mathbf{A}))}$$

**Proof of (3): column operations preserve the dimension of the column space**

4. For the column operation $\mathbf{AQ}$, since $\mathbf{Q}$ is invertible, the image of the invertible matrix equals its domain, that is, $T_{\mathbf{Q}}[\mathbb{K}^n] = \mathbb{K}^n$

5. Therefore
   $\text{Col}(\mathbf{AQ}) = \{(\mathbf{AQ})\mathbf{y} \mid \mathbf{y} \in \mathbb{K}^n\} = \{\mathbf{A}\mathbf{x} \mid \mathbf{x} \in \mathbb{K}^n\} = \text{Col}(\mathbf{A})$

6. Hence
   $\underline{\dim(\text{Col}(\mathbf{AQ})) = \dim(\text{Col}(\mathbf{A}))}$

**Proof of (2) and (4): using the transpose relation**

7. By the transpose relation ({prf:ref}`pro-row-col-duality`), we have
   - $\text{Row}(\mathbf{A}) = \text{Col}(\mathbf{A}^{\top})$
   - $\text{Row}(\mathbf{PA}) = \text{Col}((\mathbf{PA})^{\top}) = \text{Col}(\mathbf{A}^{\top}\mathbf{P}^{\top})$
   - $\text{Row}(\mathbf{AQ}) = \text{Col}((\mathbf{AQ})^{\top}) = \text{Col}(\mathbf{Q}^{\top}\mathbf{A}^{\top})$

8. Since $(\mathbf{P}^{-1})^{\top} = (\mathbf{P}^{\top})^{-1}$ and $(\mathbf{Q}^{-1})^{\top} = (\mathbf{Q}^{\top})^{-1}$, the matrices $\mathbf{P}^{\top}$ and $\mathbf{Q}^{\top}$ are also invertible

9. Applying the already proved (1) and (3) to the transposed matrices:

    
    $$\underline{\dim(\text{Row}(\mathbf{PA})) = \dim(\text{Col}(\mathbf{A}^{\top}\mathbf{P}^{\top})) = \dim(\text{Col}(\mathbf{A}^{\top})) = \dim(\text{Row}(\mathbf{A}))}$$
    
    $$\underline{\dim(\text{Row}(\mathbf{AQ})) = \dim(\text{Col}(\mathbf{Q}^{\top}\mathbf{A}^{\top})) = \dim(\text{Col}(\mathbf{A}^{\top})) = \dim(\text{Row}(\mathbf{A}))}$$

**Conclusion**

10. Since $\text{rank}(\mathbf{A}) := \dim(\text{Col}(\mathbf{A}))$, we obtain
    $\underline{\text{No invertible matrix operation changes the rank of a matrix}}$

**Geometric interpretation.** In Chapters 9 and 11 we will show that an invertible transformation is a composition of "invertible stretches, rotations, and reflections"; these operations do not change the "essential structure" of a subspace, so the dimension is preserved. This is also one of the reasons why invertible transformations are called "isomorphisms."
:::

#### Summary of Elementary Matrix Operations

| Type of operation | Matrix representation | Effect on the dimension of the row space | Effect on the dimension of the column space | Effect on the rank |
|---------|---------|----------------|----------------|-----------|
| Row scaling | $\mathbf{E}_i(c)\mathbf{A}$ | None | None | None |
| Row exchange | $\mathbf{P}_{ij}\mathbf{A}$ | None | None | None |
| Row elimination | $\mathbf{E}_{ij}(c)\mathbf{A}$ | None | None | None |
| Column scaling | $\mathbf{A}\mathbf{E}_j(c)$ | None | None | None |
| Column exchange | $\mathbf{A}\mathbf{P}_{ij}$ | None | None | None |
| Column elimination | $\mathbf{A}\mathbf{E}_{ji}(c)$ | None | None | None |

**The core property of elementary matrix operations is that they preserve the dimensions of the row space and the column space, and hence the rank of the matrix.** This property is the foundation of Gaussian elimination and the other algorithms for solving systems of linear equations. Row operations apply an invertible linear transformation on the output side and leave the solution set unchanged, which makes them the main tool for solving systems of linear equations; column operations have important applications in matrix decompositions and in understanding the structure of the solution space. Understanding how elementary operations affect the subspace structure of a matrix is the key to mastering the core ideas of linear algebra.

:::{admonition} §6.1 Summary
:class: tip

This section answered a fundamental question: do the elementary operations applied to a matrix destroy the essential structure of the system of equations? The answer is no, but the reasons deserve careful reflection. The three row operations—row scaling, row exchange, and row elimination—are each equivalent to left multiplication by an invertible elementary matrix, and so they are all invertible linear transformations; while they change the "outward form" of the system, they change neither the solution set nor the dimensions of the row space and the column space, and therefore not the rank of the matrix. Column operations (right multiplication by elementary matrices) likewise preserve the rank, but they may change the solution set of the system, so they are used mainly for matrix decompositions and the analysis of the kernel rather than for solving directly. The theorem on the preservation of subspace dimensions gives the most general statement: left or right multiplication by any invertible matrix preserves the dimensions of the row space and the column space. This conclusion is the theoretical foundation of Gaussian elimination—every step of the elimination can be applied with confidence, because what it changes is only the "representation" of the matrix, not its "essence." §6.2 will use this guarantee to bring any matrix to the standard row echelon form and read off the structure of the solutions directly from it.
:::

## 6.2 Reduced Row Echelon Form and the Geometric Structure of Solutions

The reduced row echelon form is not only a key intermediate step in solving systems of linear equations; it also gives an intuitive expression of the geometric structure of the solutions. Through the reduced row echelon form we can clearly see the dimension and shape of the solution space of a system and its relation to the subspaces of the coefficient matrix.

### 6.2.1 Square Upper Triangular Matrices with Nonzero Diagonal and Their Geometric Interpretation

**An upper triangular matrix with nonzero diagonal entries** is like a "staircase" system of equations: each equation is "simpler" than the one before it, so we can start from the last equation and solve backward step by step. In §5.3.3, {prf:ref}`thm-triangular-matrix-invertible`, we already proved that such a system necessarily has a solution; now we work through an actual example and solve it.

:::{prf:example} Solving a Three-Dimensional Upper Triangular System
:label: ex-upper-triangular
Consider the upper triangular system

$$
\begin{bmatrix}
2 & 3 & 1 \\
0 & 4 & -2 \\
0 & 0 & 3
\end{bmatrix}
\begin{bmatrix}
x_0 \\
x_1 \\
x_2
\end{bmatrix} =
\begin{bmatrix}
5 \\
6 \\
9
\end{bmatrix}
$$

**Why is this system easy to solve?**

Look at the structure of the system:
- Equation 2: $3x_2 = 9$ → contains only one variable, $x_2$, so $x_2 = 3$ directly
- Equation 1: $4x_1 - 2x_2 = 6$ → with $x_2 = 3$ known, only one unknown, $x_1$, remains
- Equation 0: $2x_0 + 3x_1 + x_2 = 5$ → with $x_1$ and $x_2$ known, only one unknown, $x_0$, remains

**Geometric interpretation.**
Imagine three planes in three-dimensional space:
- Plane 2, $3x_2 = 9$, is perpendicular to the $x_2$-axis, fixed at the position $x_2 = 3$
- Plane 1, $4x_1 - 2x_2 = 6$, intersects plane 2 in a line
- Plane 0, $2x_0 + 3x_1 + x_2 = 5$, meets the line of intersection of the first two planes in a point

This is like **narrowing the search step by step**:
three-dimensional space → two-dimensional plane → one-dimensional line → zero-dimensional point

**The solution process** (back substitution):
1. $x_2 = 3$
2. $4x_1 - 2(3) = 6 \Rightarrow x_1 = 3$  
3. $2x_0 + 3(3) + 3 = 5 \Rightarrow x_0 = -\frac{7}{2}$
:::

**Why do upper triangular matrices have this special structure?**

From the viewpoint of transformations, an upper triangular matrix has a kind of "hierarchy":
- It maps the standard basis vectors $\{\mathbf{e}_0, \mathbf{e}_1, \ldots, \mathbf{e}_{n-1}\}$ to an image with a special structure
- This image can be decomposed into nested, properly contained subspaces: $\{0\} \subset \text{span}\{\mathbf{e}_0\} \subset \text{span}\{\mathbf{e}_0,\mathbf{e}_1\} \subset \cdots \subset \mathbb{R}^n$. This is because
    - every diagonal entry is nonzero, which guarantees that every standard basis vector makes a "new contribution" to the image
    - the matrix has full rank (rank = n), so the image is the whole of $\mathbb{R}^n$
    - each additional basis vector increases the dimension of the image by 1

**An intuitive analogy.** It is like the floors of a building: the upper floors can affect the lower floors, but the lower floors cannot affect the upper ones. This "one-way dependence" lets us start from the "top floor" and solve downward floor by floor.

This structure is especially important in numerical computation, because it guarantees the stability and efficiency of the solution process.

---

### 6.2.2 Row Echelon Matrices and the Reduced Row Echelon Form

A square upper triangular matrix whose diagonal entries are all nonzero is a REF whose pivots all lie on the diagonal; a general upper triangular matrix need not be a REF.

:::{prf:definition} Row Echelon Matrix
:label: def-ref-matrix
A matrix $\mathbf{R}$ is in row echelon form if it satisfies:
1. All zero rows (if any) are at the bottom of the matrix
2. The column position of the first nonzero entry of each nonzero row (called the **pivot**) is strictly greater than the column position of the pivot of the row above
3. The entries below each pivot are all zero
:::

A matrix in row echelon form (Row Echelon Form, REF) is a special kind of upper triangular matrix (conversely, an upper triangular matrix need not be in row echelon form); its image can be decomposed into nested subspaces, but the containments are not necessarily proper.
Let us next consider a simpler form, which can be converted to and from a row echelon matrix by row scaling and row elimination: the reduced row echelon matrix (Reduced Row Echelon Form, RREF), which simplifies this geometric representation further.

:::{prf:definition} Reduced Row Echelon Matrix
:label: def-rref-matrix
A matrix $\mathbf{R}$ is in reduced row echelon form if it is in row echelon form and satisfies:
1. Every pivot equals 1
2. All other entries in the column of each pivot are 0
:::

:::{prf:example} A Row Echelon Matrix and a Reduced Row Echelon Matrix
:label: ex-ref-rref-comparison
$\mathbf{A}_{REF}=\begin{bmatrix}
1& 2 & 2 & 3 \\
0& 0 & 2 & -2 \\
0& 0 & 0 & 0
\end{bmatrix}$

$\mathbf{A}_{RREF}=\begin{bmatrix}
1& 2 & 0 & 5 \\
0& 0 & 1 & -1 \\
0& 0 & 0 & 0
\end{bmatrix}$

It is easy to see that $\mathbf{A}_{RREF} = \mathbf{E}_{01}(-2) \mathbf{E}_{1}(0.5) \mathbf{A}_{REF}$
:::

::::{exercise} The Precise Relation Between Row Echelon Form and Upper Triangular Form
:label: exer-ch6-ref-vs-upper-triangular
Prove that a row echelon matrix is upper triangular, and give a counterexample showing that an upper triangular matrix need not be a row echelon matrix.

:::{solution} exer-ch6-ref-vs-upper-triangular
:class: dropdown
**Proof (row echelon $\Rightarrow$ upper triangular).** Let $\mathbf{R}$ be in row echelon form, and let the pivot of its $i$-th nonzero row lie in column $c_i$. By definition the column positions of the pivots are strictly increasing: $c_0 < c_1 < \cdots$. From $c_0 \geq 0$ and strict increase, induction gives $c_i \geq i$. Moreover, the pivot is the first nonzero entry of its row, that is, $r_{ij} = 0$ when $j < c_i$. Hence when $i > j$, we have $j < i \leq c_i$, so $r_{ij} = 0$; the entries of the zero rows are all zero as well. Therefore all entries below the main diagonal are zero, and $\mathbf{R}$ is an upper triangular matrix. $\square$

**Counterexample (upper triangular $\not\Rightarrow$ row echelon).** Take

$$\begin{bmatrix} 0 & 1 \\ 0 & 2 \end{bmatrix}$$

It is an upper triangular matrix (the entries below the diagonal are zero), but the pivots of row 0 and row 1 both lie in column 1, so the column positions of the pivots are not strictly increasing (and there is a nonzero entry below a pivot); hence it is not a row echelon matrix.
:::
::::

:::{prf:property} Converting a Row Echelon Matrix to a Reduced Row Echelon Matrix
:label: pro-ref-to-rref
Every row echelon matrix $\mathbf{A}_{REF}$ can be turned into a reduced row echelon matrix $\mathbf{A}_{RREF}$ by row scaling and row elimination operations; hence, by the dimension preservation under elementary operations ({prf:ref}`thm-subspace-dim-preservation`), $Rank(\mathbf{A}_{REF})=Rank(\mathbf{A}_{RREF})$
:::

### 6.2.3 Reading the Rank and the Subspace Dimensions from the Block Structure of the RREF

§6.2.1 showed that an upper triangular matrix with nonzero diagonal has a nested "flag" structure of subspaces: each new pivot contributes an entirely new direction to the image, and back substitution solves step by step from the last row upward. The RREF and the upper triangular matrix share essentially the same "staircase of pivots" geometry; the only difference is that an upper triangular matrix allows nonzero entries above the pivots, while the RREF further requires the entries above and below each pivot to be cleared to zero and each pivot to equal 1. This extra reduction costs very little (row elimination plus row scaling), but it brings a decisive benefit: **the block structure of the RREF can be read off directly, with no back substitution needed**.

However, the pivots of an RREF do not fall neatly on the diagonal as they do for a square upper triangular matrix. Take the following matrix as an example:

$$
\mathbf{R}_{RREF} = \begin{bmatrix}
1 & 0 & 2 & 0 & 5 \\
0 & 1 & 3 & 0 & 4 \\
0 & 0 & 0 & 1 & -2 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

The pivots are in columns 0, 1, and 3, while columns 2 and 4 have no pivot. This phenomenon of "pivots skipping columns" tells us that **the order of the variables needs to be adjusted** before the RREF can be rearranged into a standard block form. Below we illustrate the whole process with a complete numerical example, and then state the theorem.

:::{prf:example} Complete Solution of the Homogeneous System $\mathbf{R}\mathbf{x}=\mathbf{0}$
:label: ex-rref-homogeneous-full

Given

$$
\mathbf{R} = \begin{bmatrix}
1 & 0 & 2 & 0 & 5 \\
0 & 1 & 3 & 0 & 4 \\
0 & 0 & 0 & 1 & -2 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

solve the homogeneous system $\mathbf{R}\mathbf{x}=\mathbf{0}$ and find a basis of the kernel.

**Step 1. Identify the pivot columns and the free-variable columns**

Scan the pivot positions row by row:

- The pivot of row 0 is in **column 0**, corresponding to the variable $x_0$
- The pivot of row 1 is in **column 1**, corresponding to the variable $x_1$
- The pivot of row 2 is in **column 3**, corresponding to the variable $x_3$
- Row 3 is entirely zero (a zero row)

Columns 2 and 4 have no pivot, so the corresponding variables $x_2, x_4$ can take any values.

| Column index | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Role | pivot variable $x_0$ | pivot variable $x_1$ | **free variable** $x_2$ | pivot variable $x_3$ | **free variable** $x_4$ |

**Step 2. Write out the system directly from the RREF**

Expand the first three nontrivial rows of $\mathbf{R}\mathbf{x}=\mathbf{0}$ (the zero row is satisfied automatically):

$$
\begin{cases}
x_0 + 2x_2 + 5x_4 = 0 \\
x_1 + 3x_2 + 4x_4 = 0 \\
x_3 - 2x_4 = 0
\end{cases}
$$

The left side of each equation contains exactly one pivot variable, and the right side is a linear combination of the free variables. This is the key property of the RREF: **the pivot variables can be expressed directly in terms of the free variables**, with no back substitution needed.

$$
\begin{cases}
x_0 = -2x_2 - 5x_4 \\
x_1 = -3x_2 - 4x_4 \\
x_3 = 2x_4
\end{cases}
$$

By setting $x_2= 1,x_4=0$ or $x_2= 0,x_4=1$ we obtain two vectors, and an easy calculation shows that they are solutions of the system $\mathbf{R}\mathbf{x}=\mathbf{0}$. But could there be other solutions? What if we have overlooked something? This is why we need to rearrange the matrix and obtain a general theory.

$$
\begin{bmatrix}
1 & 0 & 2 & 0 & 5 \\
0 & 1 & 3 & 0 & 4 \\
0 & 0 & 0 & 1 & -2 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix} \begin{bmatrix} -2 \\ -3 \\ 1 \\ 0 \\ 0 \end{bmatrix} =  \begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \end{bmatrix}
$$

$$
\begin{bmatrix}
1 & 0 & 2 & 0 & 5 \\
0 & 1 & 3 & 0 & 4 \\
0 & 0 & 0 & 1 & -2 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}  \begin{bmatrix} -5 \\ -4 \\ 0 \\ 2 \\ 1 \end{bmatrix} =\begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \end{bmatrix} 
$$


**Step 3. Permutation: move the free variables to the end**

To write the structure of the solutions in a tidy block form, use a column permutation matrix $\mathbf{P}$ (multiplying $\mathbf{R}$ on the right) to rearrange the columns into the order "pivot-variable columns first, free-variable columns last": the new order is $[x_0, x_1, x_3, x_2, x_4]$, that is, columns 2 and 4 are moved to the end.

$$
\mathbf{R}\,\mathbf{P} = \begin{bmatrix}
1 & 0 & 0 & 2 & 5 \\
0 & 1 & 0 & 3 & 4 \\
0 & 0 & 1 & 0 & -2 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}
= \begin{bmatrix} \mathbf{I}_3 & \mathbf{F} \\ \mathbf{0} & \mathbf{0} \end{bmatrix}
$$

where

$$
\mathbf{F} = \begin{bmatrix} 2 & 5 \\ 3 & 4 \\ 0 & -2 \end{bmatrix}
$$

The permuted matrix acts on the rearranged vector $\mathbf{y} = \mathbf{P}^{-1}\mathbf{x} = \mathbf{P}^\top\mathbf{x}$, whose components are in the order $[x_0, x_1, x_3, x_2, x_4]^\top$; this can also be written $\mathbf{y} = [\mathbf{y}_p^\top, \mathbf{y}_f^\top]^\top$, where $\mathbf{y}_p = [x_0, x_1, x_3]^\top$ are the pivot variables and $\mathbf{y}_f = [x_2, x_4]^\top$ are the free variables. Note that the variables themselves are not basis vectors: the **positions** of the pivot variables correspond to the pivot columns, and these column vectors form a basis of the image; the **number** of free variables equals the dimension of the kernel, and each free variable corresponds to one vector of a basis of the kernel (see the columns of $\mathbf{N}$ in Steps 4–6).

**Step 4. Solve in blocks**

The block expansion of the system $(\mathbf{R}\mathbf{P})\mathbf{y} = \mathbf{0}$ is

$$
\begin{bmatrix} \mathbf{I}_3 & \mathbf{F} \\ \mathbf{0} & \mathbf{0} \end{bmatrix} \begin{bmatrix} \mathbf{y}_p \\ \mathbf{y}_f \end{bmatrix} = \mathbf{0}
\quad\Rightarrow\quad
\mathbf{y}_p = -\mathbf{F}\,\mathbf{y}_f
$$

Let $\mathbf{y}_f = [t_0, t_1]^\top$ (with $t_0, t_1 \in \mathbb{R}$ arbitrary); then

$$
\mathbf{y} = \begin{bmatrix} -\mathbf{F}\,\mathbf{y}_f \\ \mathbf{y}_f \end{bmatrix}
= \begin{bmatrix} -\mathbf{F} \\ \mathbf{I}_2 \end{bmatrix} \begin{bmatrix} t_0 \\ t_1 \end{bmatrix}
= t_0 \begin{bmatrix} -2 \\ -3 \\ 0 \\ 1 \\ 0 \end{bmatrix} + t_1 \begin{bmatrix} -5 \\ -4 \\ 2 \\ 0 \\ 1 \end{bmatrix}
$$

Note that the components here are in the order $[x_0, x_1, x_3, x_2, x_4]$.

**Step 5. Restore the original order of the variables**

Use $\mathbf{P}$ to restore $\mathbf{x} = \mathbf{P}\mathbf{y}$, moving $x_3$ back to component 3 and $x_2$ back to component 2:

$$
\mathbf{x} = \begin{bmatrix} x_0 \\ x_1 \\ x_2 \\ x_3 \\ x_4 \end{bmatrix}
= t_0 \begin{bmatrix} -2 \\ -3 \\ 1 \\ 0 \\ 0 \end{bmatrix}
+ t_1 \begin{bmatrix} -5 \\ -4 \\ 0 \\ 2 \\ 1 \end{bmatrix},
\quad t_0, t_1 \in \mathbb{R}
$$

**Step 6. Check with the rank–nullity theorem**

By the rank–nullity theorem of Chapter 4, §4.2.6 ({prf:ref}`thm-rank-nullity`), for any linear mapping $T: V \to W$,

$$
\dim(\ker T) + \dim(\operatorname{Im} T) = \dim V
$$

Applied to $T_\mathbf{R}: \mathbb{R}^5 \to \mathbb{R}^4$:

$$
\dim(\ker \mathbf{R}) + \operatorname{rank}(\mathbf{R}) = 5
$$

Reading $\operatorname{rank}(\mathbf{R}) = 3$ directly from the RREF (there are 3 pivots), we get $\dim(\ker \mathbf{R}) = 5 - 3 = 2$.

The two kernel vectors found above are linearly independent (their free-variable coefficients form $\mathbf{I}_2$), so they do form a basis of the kernel, in agreement with the prediction of the theorem; we have not missed anything.

**Step 7. Python numerical verification**
:::

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ──────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

# ============================================================
print_header("Example 6.1 | Solving a Homogeneous RREF System and Verifying the Kernel")
# ============================================================

# --- Step 1: define the RREF matrix R ---
print_step(1, "Define the RREF matrix R")
R = np.array([
    [1, 0, 2, 0,  5],
    [0, 1, 3, 0,  4],
    [0, 0, 0, 1, -2],
    [0, 0, 0, 0,  0]
], dtype=float)
print("R =\n", R)

# --- Step 2: read a basis of the kernel directly from the block structure ---
print_step(2, "Basis of the kernel (read directly from the free-variable columns)")
# pivot columns: 0, 1, 3  ;  free columns: 2, 4
# assign unit values to the free variables (x₂, x₄) in turn
n0 = np.array([-2, -3, 1, 0, 0], dtype=float)   # x₂ = 1, x₄ = 0
n1 = np.array([-5, -4, 0, 2, 1], dtype=float)   # x₂ = 0, x₄ = 1
print("n₀ =", n0)
print("n₁ =", n1)

# --- Step 3: check R @ nᵢ = 0 ---
print_step(3, "Check R·nᵢ = 0 (nᵢ belongs to Null(R))")
compare_print("R · n₀", R @ n0, "zero vector [0, 0, 0, 0]")
compare_print("R · n₁", R @ n1, "zero vector [0, 0, 0, 0]")

# --- Step 4: check the dimensions with the rank–nullity theorem ---
print_step(4, "Check with the rank–nullity theorem")
rank_R    = np.linalg.matrix_rank(R)
nullity_R = R.shape[1] - rank_R
compare_print("rank(R)",    rank_R,    "3 (three pivot columns)")
compare_print("nullity(R)", nullity_R, "2 (two free columns)")
print(f"\n  rank(R) + nullity(R) = {rank_R} + {nullity_R} = {rank_R + nullity_R}"
      f"  = n = {R.shape[1]}  ✓")

With this concrete example as a foundation, we can now state the general theorem.

:::{prf:theorem} Block Structure Theorem for the RREF (Coordinate Version of Rank–Nullity)
:label: thm-rref-rank-block

Let $\mathbf{R}$ be an $m \times n$ matrix in RREF with $r$ pivots. Let the $r$ columns containing the pivots be $c_0 < c_1 < \cdots < c_{r-1}$; the remaining $n - r$ columns are the free-variable columns. Let $\mathbf{P}$ be the permutation matrix that rearranges these columns into the order "pivot columns first, free-variable columns last." Then

$$
\mathbf{R}\,\mathbf{P} = \begin{bmatrix} \mathbf{I}_r & \mathbf{F} \\ \mathbf{0} & \mathbf{0} \end{bmatrix}
$$

where $\mathbf{F}$ is an $r \times (n-r)$ matrix. From this we can read off directly:

1. $\operatorname{rank}(\mathbf{R}) =\dim(\operatorname{Col}(\mathbf{R})) = r$ (the number of pivots)
2. $\dim(\operatorname{Row}(\mathbf{R})) = r$ (the dimension of the row space)
3. $\dim(\ker(\mathbf{R})) = n - r$ (the dimension of the kernel), and the kernel is spanned by the following $n - r$ vectors:
   $$\mathbf{N} = \mathbf{P}\begin{bmatrix} -\mathbf{F} \\ \mathbf{I}_{n-r} \end{bmatrix}$$
4. $\dim(\ker(\mathbf{R}^\top)) = m - r$ (the dimension of the left null space)
:::

:::{prf:proof}
:label: prf-rref-rank-block


1. The first $r$ columns of $\mathbf{R}\mathbf{P}$ are the standard basis vectors $\mathbf{e}_0, \ldots, \mathbf{e}_{r-1} \in \mathbb{R}^m$, which are linearly independent; the last $n-r$ columns are linear combinations of these vectors (formed from the columns of $\mathbf{F}$). Elementary matrix operations preserve the dimension of the column space, so $\underline{\operatorname{rank}(\mathbf{R}) = r}$.

2. The nonzero rows of $\mathbf{R}\mathbf{P}$ are exactly the $r$ row vectors of $[\mathbf{I}_r \;\, \mathbf{F}]$: their first $r$ components together form $\mathbf{I}_r$, so they are linearly independent and $\dim(\operatorname{Row}(\mathbf{R}\mathbf{P})) = r$. Moreover, right multiplication by an invertible matrix preserves the dimension of the row space ({prf:ref}`thm-subspace-dim-preservation`), so $\underline{\dim(\operatorname{Row}(\mathbf{R})) = r}$

3. Verifying the kernel: for any $\mathbf{y}_f \in \mathbb{R}^{n-r}$, let $\mathbf{y} = \begin{bmatrix} -\mathbf{F}\mathbf{y}_f \\ \mathbf{y}_f \end{bmatrix}$; then
   $$(\mathbf{R}\mathbf{P})\mathbf{y} = \begin{bmatrix}\mathbf{I}_r(-\mathbf{F}\mathbf{y}_f) + \mathbf{F}\mathbf{y}_f\\\mathbf{0}_{m-r}\end{bmatrix} = \mathbf{0}_m$$
   so $\mathbf{y} \in \ker(\mathbf{R}\mathbf{P})$. Conversely, if $(\mathbf{R}\mathbf{P})\mathbf{y} = \mathbf{0}$, then $\mathbf{y}_p = -\mathbf{F}\mathbf{y}_f$, which shows that every kernel vector has this form. The $n - r$ basis vectors (obtained by taking $\mathbf{y}_f$ to be the standard basis vectors) are linearly independent, so $\dim(\ker(\mathbf{R}\mathbf{P})) = n - r$. Moreover, $\ker(\mathbf{R}\mathbf{P}) = \mathbf{P}^{-1}\ker(\mathbf{R}) = \mathbf{P}^\top \ker(\mathbf{R})$, so $\underline{\dim(\ker(\mathbf{R})) = \dim(\ker(\mathbf{R}\mathbf{P})) = n-r}$.

4. Since $\dim(\operatorname{Col}(\mathbf{R})) = \dim(\operatorname{Row}(\mathbf{R}))= \dim(\operatorname{Col}(\mathbf{R}^\top)) = \dim(\operatorname{Row}(\mathbf{R}^\top))  = r $ and the transpose is an $n \times m$ matrix, the rank–nullity theorem gives $\underline{\dim(\ker(\mathbf{R}^\top)) = m - r}$. $\square$
:::

:::{prf:remark} The Equality of Row-Space and Column-Space Dimensions in General
:label: rem-row-col-dim-equal

Theorem {prf:ref}`thm-rref-rank-block` reads off directly from the block structure, under the hypothesis that "the matrix is already in RREF," that the dimensions of the row space and the column space are both $r$. For a general matrix, §6.3 will prove that every $m \times n$ matrix $\mathbf{A}$ has a PLU decomposition

$$\mathbf{PA} = \mathbf{L}\mathbf{U}$$

where $\mathbf{P}$ is a permutation matrix (invertible), $\mathbf{L}$ is a unit lower triangular matrix (invertible), and $\mathbf{U}$ is a row echelon matrix. By the dimension-preservation theorem for elementary operations of §6.1, left multiplication by an invertible matrix does not change the dimensions of the row space and the column space, so

$$\dim(\operatorname{Row}(\mathbf{A})) = \dim(\operatorname{Row}(\mathbf{U})) = \dim(\operatorname{Col}(\mathbf{U})) = \dim(\operatorname{Col}(\mathbf{A}))$$

After $\mathbf{U}$ is further row reduced to RREF, this theorem applies; invertible row operations preserve the dimensions of both the row space and the column space, which gives the middle equality. This shows that for any matrix, **the dimension of the row space always equals the dimension of the column space**, and both equal the rank $r$.
:::

### 6.2.4 Nonhomogeneous Systems of Linear Equations: Particular Solution plus Homogeneous Solution

§6.2.3 settled the homogeneous system $\mathbf{R}\mathbf{x}=\mathbf{0}$: the structure of the kernel is completely determined by the block matrix $\begin{bmatrix}\mathbf{I}_r & \mathbf{F}\end{bmatrix}$, and the basis matrix is $\mathbf{N} = \mathbf{P}\begin{bmatrix}-\mathbf{F}\\\mathbf{I}_{n-r}\end{bmatrix}$.

Now consider the general case $\mathbf{A}\mathbf{x}=\mathbf{b}$, $\mathbf{b}\neq\mathbf{0}$. Compared with the homogeneous case, a nonhomogeneous system faces a new problem: **a solution need not exist**. Before discussing how to solve, we must first ask: what condition must $\mathbf{b}$ satisfy for the system to have a solution?

#### Existence of Solutions: Two Views

**The hyperplane view.** Each row of the system $\mathbf{A}\mathbf{x}=\mathbf{b}$ is the equation of a hyperplane in $\mathbb{R}^n$. Having a solution is equivalent to all the hyperplanes having a common point of intersection. If the RREF of the augmented matrix $[\mathbf{R}|\mathbf{c}]$ contains a row of the form

$$[0,\,0,\,\ldots,\,0\;|\;k],\quad k\neq 0$$

then that row requires "$0 = k$," which cannot be satisfied by any $\mathbf{x}$; this shows that the hyperplanes contradict one another and have no common point of intersection.

**The image view.** Regard $\mathbf{A}\mathbf{x}=\mathbf{b}$ as asking: does $\mathbf{b}$ lie in the column space (the image) $\operatorname{Col}(\mathbf{A})$ of $\mathbf{A}$?

$$\mathbf{A}\mathbf{x}=\mathbf{b} \text{ has a solution} \leftrightarrow \mathbf{b}\in\operatorname{Col}(\mathbf{A})$$

Because $\mathbf{A}\mathbf{x}$ is a linear combination of the columns of $\mathbf{A}$, having a solution is equivalent to $\mathbf{b}$ being expressible as a linear combination of these columns. In terms of rank: $\mathbf{b}\in\operatorname{Col}(\mathbf{A})$ if and only if appending $\mathbf{b}$ to $\mathbf{A}$ does not increase the dimension of the column space, that is,

$$\operatorname{rank}([\mathbf{A}|\mathbf{b}]) = \operatorname{rank}(\mathbf{A})$$

The two views are equivalent: an inconsistent row appearing in the RREF is equivalent to $\mathbf{b}$ not lying in the image of $\mathbf{A}$, which is equivalent to the rank of the augmented matrix exceeding the rank of the coefficient matrix.

#### The Structure of the Solutions When a Solution Exists: Particular Solution plus Homogeneous Solution

Once we have confirmed that the system has a solution, the structure of the solutions is described by the following theorem:

:::{prf:theorem} Structure of the Solutions of a Nonhomogeneous System
:label: thm-nonhomogeneous-solution-structure

Suppose $\mathbf{A}\mathbf{x}=\mathbf{b}$ has a solution, and let $\mathbf{x}_p$ be any particular solution. Then **all the solutions** of the system are

$$\mathbf{x} = \mathbf{x}_p + \mathbf{x}_h,\quad \mathbf{x}_h\in\ker(\mathbf{A})$$

that is, the solution set is $\mathbf{x}_p + \ker(\mathbf{A})$, an affine subspace obtained by translating the kernel, of dimension $n - \operatorname{rank}(\mathbf{A})$.
:::

:::{prf:proof}
1. If $\mathbf{A}\mathbf{x}_p = \mathbf{b}$ and $\mathbf{x}_h \in \ker(\mathbf{A})$, then $\mathbf{A}(\mathbf{x}_p + \mathbf{x}_h) = \mathbf{b} + \mathbf{0} = \mathbf{b}$, so $\mathbf{x}_p+\mathbf{x}_h$ is also a solution.
2. Conversely, if $\mathbf{x}$ is any solution, let $\mathbf{x}_h = \mathbf{x}-\mathbf{x}_p$; then $\mathbf{A}\mathbf{x}_h = \mathbf{A}\mathbf{x}-\mathbf{A}\mathbf{x}_p = \mathbf{b}-\mathbf{b} = \mathbf{0}$, so $\mathbf{x}_h\in\ker(\mathbf{A})$.

$\underline{\text{solution set} = \mathbf{x}_p + \ker(\mathbf{A})\text{, of dimension }n-\operatorname{rank}(\mathbf{A})}$
:::

Let $\mathbf{R}=\mathbf{E}\mathbf{A}$; the right-hand side must undergo the same row operations, giving $\mathbf{c}=\mathbf{E}\mathbf{b}$. Let $\mathbf{b}_p$ be the first $r$ components of $\mathbf{c}$; consistency requires the remaining $m-r$ components all to be zero. After the columns are rearranged we have $\mathbf{y}_p=\mathbf{b}_p-\mathbf{F}\mathbf{y}_f$. In this block structure the most natural choice of particular solution is to set all the free variables to zero ($\mathbf{y}_f = \mathbf{0}$); then $\mathbf{y}_p = \mathbf{b}_p$, and restoring the order of the columns gives a particular solution. The complete solution is then obtained by adding to this particular solution the kernel already found in §6.2.3.

:::{prf:example} A Nonhomogeneous RREF System: Existence, a Particular Solution, and the Homogeneous Solutions
:label: ex-nonhomogeneous-rref-full

Keeping the coefficient matrix of §6.2.3, consider the following two right-hand-side vectors:

**(a) The case with a solution:** $\mathbf{b} = [7, 11, 6, 0]^\top$

The RREF of the augmented matrix is

$$
[\mathbf{R}|\mathbf{b}] = \begin{bmatrix}
1 & 0 & 2 & 0 & 5 & | & 7 \\
0 & 1 & 3 & 0 & 4 & | & 11 \\
0 & 0 & 0 & 1 & -2 & | & 6 \\
0 & 0 & 0 & 0 & 0 & | & 0
\end{bmatrix}
$$

**Step 1. Test for existence**

Row 3 (the last row) is $[0,0,0,0,0\,|\,0]$, so there is no contradiction. $\operatorname{rank}(\mathbf{R}) = \operatorname{rank}([\mathbf{R}|\mathbf{b}]) = 3$, so the system has a solution.

Interpretation in terms of the image: $\mathbf{b}=[7,11,6,0]^\top\in\mathbb{R}^4$ lies in the three-dimensional image of $\mathbf{R}$; $\mathbf{b}_p=[7,11,6]^\top\in\mathbb{R}^3$ gives the right-hand-side coordinates after the zero equation at the bottom is deleted.

**Step 2. Choose a particular solution**

After the permutation, the block equation is $\mathbf{y}_p = \mathbf{b}_p - \mathbf{F}\mathbf{y}_f$. Setting the free variables $x_2 = x_4 = 0$ (that is, $\mathbf{y}_f = \mathbf{0}$) gives the particular solution

$$
\mathbf{x}_p = \begin{bmatrix}7\\11\\0\\6\\0\end{bmatrix}
$$

**Step 3. Add the homogeneous solutions**

By §6.2.3, a basis of $\ker(\mathbf{R})$ is

$$
\mathbf{n}_0 = \begin{bmatrix}-2\\-3\\1\\0\\0\end{bmatrix},\quad
\mathbf{n}_1 = \begin{bmatrix}-5\\-4\\0\\2\\1\end{bmatrix}
$$

The complete solution is

$$
\mathbf{x} = \underbrace{\begin{bmatrix}7\\11\\0\\6\\0\end{bmatrix}}_{\text{particular solution}} + t_0\begin{bmatrix}-2\\-3\\1\\0\\0\end{bmatrix} + t_1\begin{bmatrix}-5\\-4\\0\\2\\1\end{bmatrix},\quad t_0,t_1\in\mathbb{R}
$$

Geometrically, this is a two-dimensional affine subspace of $\mathbb{R}^5$—the kernel through the origin (a two-dimensional plane), translated so that it passes through the particular solution $\mathbf{x}_p$.

**(b) The case with no solution:** $\mathbf{b}' = [7, 11, 6, 3]^\top$

The echelon form obtained by eliminating in the augmented matrix contains

$$
[\mathbf{R}|\mathbf{b}'] = \begin{bmatrix}
1 & 0 & 2 & 0 & 5 & | & \ast \\
0 & 1 & 3 & 0 & 4 & | & \ast \\
0 & 0 & 0 & 1 & -2 & | & \ast \\
0 & 0 & 0 & 0 & 0 & | & 3
\end{bmatrix}
$$

Row 3 requires $0 = 3$, a contradiction. $\operatorname{rank}([\mathbf{R}|\mathbf{b}']) = 4 > 3 = \operatorname{rank}(\mathbf{R})$, so the system has no solution.

Interpretation in terms of the image: component 3 of $\mathbf{b}'$ (corresponding to the zero row) is not zero, which shows that $\mathbf{b}'$ lies outside the image of $\mathbf{A}$ and cannot be expressed as any $\mathbf{A}\mathbf{x}$.

:::

The three forms of the solutions are now clear at a glance:

| Case | Condition (RREF) | Solution set | Dimension |
|---|---|---|---|
| No solution | An inconsistent row $[0\cdots0\,\|\,k]$, $k\neq 0$, appears | Empty set | — |
| Unique solution | No inconsistent row, and no free variables ($r=n$) | A single point $\{\mathbf{x}_p\}$ | 0 |
| Infinitely many solutions | No inconsistent row, and there are free variables ($r<n$) | The affine subspace $\mathbf{x}_p+\ker(\mathbf{A})$ | $n-r$ |

:::{admonition} §6.2 Summary
:class: tip

This section answered the central question "what exactly is the structure of the solutions of a system of equations?" The reduced row echelon form (RREF) is the end point of elimination: by a suitable column permutation, any RREF matrix can be partitioned into blocks as $\begin{bmatrix}\mathbf{I}_r & \mathbf{F} \\ \mathbf{0} & \mathbf{0}\end{bmatrix}$, where $r$ is the rank and the matrix $\mathbf{F}$ describes the linear dependence $\mathbf{y}_p = -\mathbf{F}\mathbf{y}_f$ between the pivot variables and the free variables. From this block structure we can read off directly the dimensions of the four subspaces: the row space and the column space both have dimension $r$, the kernel has dimension $n-r$, and the left null space has dimension $m-r$—this is the concrete source of the rank–nullity theorem that follows. The three forms of the solutions also become clear at a glance here: the solution is unique when the system is consistent and has no free variables; there is no solution when the augmented matrix contains an inconsistent row; and otherwise the solution set is an $(n-r)$-dimensional affine subspace. Here the rank receives its most direct computational definition, and it formally connects with the concept of dimension from Chapter 4 on abstract linear spaces. §6.3 will explain how to bring a general matrix systematically to REF or RREF, namely Gaussian elimination and its matrix language—the LU decomposition and the PLU decomposition.
:::

## 6.3 Gaussian Elimination

Gaussian elimination is the systematic method for transforming the augmented matrix of a system of linear equations into row echelon form (REF) or reduced row echelon form (RREF). It provides the actual computational steps for reaching the matrix forms discussed in §6.2.

### 6.3.1 The Essence of Gaussian Elimination

At the heart of Gaussian elimination is a sequence of **elementary row operations** that transform the augmented matrix into an equivalent row echelon matrix. These operations are in essence **isomorphic transformations of the image**, which guarantee that the solution set of the system is the same before and after the transformation.

:::{prf:property} The Isomorphism Property of Gaussian Elimination
:label: pro-gaussian-essence
The key features of Gaussian elimination:
1. It uses only elementary row operations (row scaling, row exchange, row elimination), and these operations are isomorphisms on the image
2. Every operation is invertible, so the solutions of the original system can be obtained by mapping back through the isomorphism
3. It changes the form in which the system is represented without changing its solution set
:::

```{important}
:label: imp-isomorphism
Algebraically, Gaussian elimination amounts to applying an invertible linear transformation $\mathbf{P}$ to the linear system, turning $\mathbf{Ax} = \mathbf{b}$ into $\mathbf{PAx} = \mathbf{Pb}$. Since $\mathbf{P}$ is invertible, the two systems are completely equivalent and have the same solution set.
```

### 6.3.2 Simplified Gaussian Elimination and the LU Decomposition


```{note}
**A note on the structure of this section.** This section first introduces the "idealized LU decomposition"—it is elegant in form when the pivots are never zero, and it provides the structural template for the PLU decomposition that follows. §6.3.3 presents its Python implementation, and §6.3.4 then introduces row exchanges to make it applicable to every matrix. Learning a simplified version with a precondition first makes the motivation for the later improvements easy to see.
```

Before giving the definition, let us first settle a question of notation: the elimination process produces a product of elementary matrices
$\mathbf{E}_{n-1} \cdots \mathbf{E}_1 \mathbf{E}_0 \,\mathbf{A} = \mathbf{U}$,
and there are two possible choices of notation:

:::{caution} Why $\mathbf{A} = \mathbf{LU}$ Rather Than $\mathbf{LA} = \mathbf{U}$
:label: cau-why-LU

If we adopt $\mathbf{LA} = \mathbf{U}$, then $\mathbf{L} = \mathbf{E}_{n-1} \cdots \mathbf{E}_1 \mathbf{E}_0$ is the product of all the elementary matrices—to obtain it, we must actually compute the product of all the matrices, or else compute an inverse once more.

By contrast, rewriting this as $\mathbf{A} = (\mathbf{E}_{n-1} \cdots \mathbf{E}_0)^{-1}\mathbf{U}$ and setting $\mathbf{L} := (\mathbf{E}_{n-1} \cdots \mathbf{E}_0)^{-1} = \mathbf{E}_0^{-1}\mathbf{E}_1^{-1}\cdots\mathbf{E}_{n-1}^{-1}$, this matrix has a striking property: **the elimination multipliers of each round can be entered directly into the corresponding positions of $\mathbf{L}$, with no matrix multiplication at all** (see the rank-one view and the worked example below). This is why the notation $\mathbf{A} = \mathbf{LU}$ is preferred.
:::

:::{prf:definition} Matrix Representation of the LU Decomposition
:label: def-lu-decomposition
Let $\mathbf{A}\in\mathbb{R}^{m\times n}$. If there exist matrices $\mathbf{L}\in\mathbb{R}^{m\times m}$ and $\mathbf{U}\in\mathbb{R}^{m\times n}$ satisfying
$$\mathbf{A}=\mathbf{LU},$$
where

- $\mathbf{L}$ is a **unit lower triangular** matrix: $L_{ii}=1$ ($0\le i<m$), and $L_{ij}=0$ when $i<j$;
- $\mathbf{U}$ is an **upper trapezoidal** matrix: $U_{ij}=0$ when $i>j$;

then $\mathbf{A}=\mathbf{LU}$ is called an **LU decomposition** of $\mathbf{A}$.

Depending on the shape of $\mathbf{A}$, the zero structure of $\mathbf{U}$ is as follows:

- $m\le n$ (the matrix is "wide"): the first $m$ columns of $\mathbf{U}$ form an $m\times m$ upper triangular square matrix, and the remaining columns are unrestricted;
- $m>n$ (the matrix is "tall"): the first $n$ rows of $\mathbf{U}$ form an $n\times n$ upper triangular square matrix, and from row $n$ on all rows are zero rows;
- $m=n$: $\mathbf{L}$ and $\mathbf{U}$ are a unit lower triangular and an upper triangular square matrix, respectively.
:::

**Core idea.** Ideal Gaussian elimination brings the original matrix $\mathbf{A}$ step by step to an upper triangular (upper trapezoidal) matrix $\mathbf{U}$ using pure "row elimination operations" alone (without interchanging the order of the rows); if the pivots fall in turn on the diagonal and are all nonzero, $\mathbf{U}$ is also in row echelon form; otherwise, in general, it is only guaranteed to be upper triangular and need not be a REF.

:::{prf:algorithm} Step-by-Step Procedure for the Problematic LU Decomposition
:label: alg-LU

**Step 1. Preparation**
- Choose a very small number (say 0.000000000001) as the criterion for deciding
- Prepare two new matrices: set the matrix $\mathbf{L}$ to the identity matrix, and let the matrix $\mathbf{U}$ be a copy of the original matrix A

**Step 2. Process each diagonal position in turn**
Starting from the upper-left corner, check the positions on the diagonal one by one:

> **For each diagonal position k:**
> 
> 1. **Check the pivot entry**
>    - If the value at the diagonal position is large enough, use it for the computation
>    - If the diagonal position is close to zero, a further check is needed
> 
> 2. **Handle the case of a zero pivot entry**
>    - Check whether this column still has nonzero numbers below the diagonal
>    - If there is a nonzero number but we are not allowed to interchange rows, declare that *the decomposition fails*
>    - If this column is all zero below, skip this position
> 
> 3. **Perform the elimination**
>    - For each row below the diagonal, compute the multiple to be eliminated
>    - Record this multiple in the matrix L
>    - Use this multiple to eliminate the number below the diagonal to zero
>    - Update the corresponding row of the matrix U

**Step 3. Finish and verify**
- After all diagonal positions have been processed, we obtain the two matrices $\mathbf{L}$ and $\mathbf{U}$
- Verify whether $\underline{\mathbf{A=LU}}$ holds

:::



:::{prf:example} A Concrete Example: LU Decomposition of a 3×3 Matrix
:label: ex-complex-lu

Compute the LU decomposition of the matrix

$$\mathbf{A} = \begin{bmatrix} 2 & 1 & 1 \\ 4 & -6 & 0 \\ -2 & 7 & 2 \end{bmatrix}$$

**Round 0 of elimination ($k=0$).**

- **Pivot entry:** $a_{00} = 2 \neq 0$, which can be used as the pivot
- **Compute the elimination multipliers:**
  - $m_{10} = \frac{a_{10}}{a_{00}} = \frac{4}{2} = 2$
  - $m_{20} = \frac{a_{20}}{a_{00}} = \frac{-2}{2} = -1$

- **Row elimination operations:**
  - Row 1 ← Row 1 − $2 \times$ Row 0: $(4, -6, 0) - 2 \times (2, 1, 1) = (0, -8, -2)$
  - Row 2 ← Row 2 − $(-1) \times$ Row 0: $(-2, 7, 2) - (-1) \times (2, 1, 1) = (0, 8, 3)$

- **Intermediate result:**

$$\begin{bmatrix} 2 & 1 & 1 \\ 0 & -8 & -2 \\ 0 & 8 & 3 \end{bmatrix}$$

**Round 1 of elimination ($k=1$).**

- **Pivot entry:** $a_{11} = -8 \neq 0$, which can be used as the pivot
- **Compute the elimination multiplier:** $m_{21} = \frac{a_{21}}{a_{11}} = \frac{8}{-8} = -1$

- **Row elimination operation:**
  - Row 2 ← Row 2 − $(-1) \times$ Row 1: $(0, 8, 3) - (-1) \times (0, -8, -2) = (0, 0, 1)$

- **The final upper triangular matrix:**

$$\mathbf{U} = \begin{bmatrix} 2 & 1 & 1 \\ 0 & -8 & -2 \\ 0 & 0 & 1 \end{bmatrix}$$

**Constructing the matrix $\mathbf{L}$.**

Enter the recorded elimination multipliers into a unit lower triangular matrix:

$$\mathbf{L} = \begin{bmatrix} 
1 & 0 & 0 \\ 
m_{10} & 1 & 0 \\ 
m_{20} & m_{21} & 1 
\end{bmatrix} = \begin{bmatrix} 
1 & 0 & 0 \\ 
2 & 1 & 0 \\ 
-1 & -1 & 1 
\end{bmatrix}$$

**Verifying that the decomposition is correct.**

Compute $\mathbf{LU}$:

$$
\begin{bmatrix} 
\textcolor{#FF8C00}{1} & 0 & 0 \\ 
\textcolor{#FF8C00}{2} & \textcolor{#006385}{1} & 0 \\ 
\textcolor{#FF8C00}{-1} & \textcolor{#006385}{-1} & \textcolor{#57068C}{1} 
\end{bmatrix}
\begin{bmatrix} 
\textcolor{#FF8C00}{2} & \textcolor{#FF8C00}{1} & \textcolor{#FF8C00}{1} \\ 
0 & \textcolor{#006385}{-8} & \textcolor{#006385}{-2} \\ 
0 & 0 & \textcolor{#57068C}{1} 
\end{bmatrix}
$$

$$
= \begin{bmatrix} 
\textcolor{#FF8C00}{2} & \textcolor{#FF8C00}{1} & \textcolor{#FF8C00}{1} \\ 
\textcolor{#FF8C00}{4} & \textcolor{#006385}{-6} & \textcolor{#006385}{0} \\ 
\textcolor{#FF8C00}{-2} & \textcolor{#006385}{7} & \textcolor{#57068C}{2} 
\end{bmatrix} = \mathbf{A}
$$

The decomposition succeeds! We have obtained $\mathbf{A} = \mathbf{LU}$.
:::

:::{danger} Problems with the LU Decomposition
1. **It need not exist.** The LU decomposition requires the pivots to fall in turn on the diagonal, with no row exchanges during the process; hence
   $$\mathbf{A}=\begin{bmatrix}0&1\\1&1\end{bmatrix}$$
   has no LU decomposition: pivot 0 is 0, so the 1 below it cannot be eliminated. (This can be checked directly: the $(0,0)$ entry of any $\mathbf{LU}$ is $U_{00}$, while the $(1,0)$ entry is $L_{10}U_{00}$, and the two cannot be 0 and 1 respectively.)
2. **Even when it exists, it need not be stable.** Change the example above to $\begin{bmatrix}10^{-12}&1\\1&1\end{bmatrix}$: the LU decomposition exists mathematically, but $\mathbf{L}$ contains a huge entry such as $10^{12}$, the errors are amplified in floating-point arithmetic, and the result cannot be trusted. This is why pivoting (row exchange) is indispensable in practice.
3. **Without row exchanges there is no way out.** If, to get around problem 1, we take some row further down as the pivot row but still do not interchange rows, then the elimination matrices are no longer lower triangular, and the two factors we end up with are neither an $\mathbf{L}$ nor a $\mathbf{U}$—this is no longer an LU decomposition, only an elimination in which the permutation was not recorded.
4. **Even when it exists, it need not be unique.** When $\mathbf{A}$ is singular, $\mathbf{U}$ has a zero pivot, and the corresponding entries of $\mathbf{L}$ can be chosen arbitrarily.
5. **Conventions on shape.** When $m>n$, the $\mathbf{U}$ defined in this book has $m-n$ zero rows at the bottom; in practice one often uses instead the "economy-size" $\mathbf{L}\in\mathbb{R}^{m\times n}$, $\mathbf{U}\in\mathbb{R}^{n\times n}$, and the two differ only by a truncation. This book adopts the $m\times m$ / $m\times n$ convention, which follows the same elimination idea as LAPACK `getrf`, but here the full $m\times m$ factor is kept, whereas libraries usually use compact storage or trapezoidal factors.

Even so, when it is feasible, the LU decomposition is simple in form and has beautiful properties; moreover, the next section will show that as long as row exchanges are allowed, every matrix has a PLU decomposition ({prf:ref}`thm-plu-existence`), in which $\mathbf{U}$ is not only upper trapezoidal but also in row echelon form. When no row exchange is needed and the pivots advance in turn, the elimination of this section can be regarded as a special case of the PLU decomposition; in a general algebraic LU decomposition, U is only required to be upper trapezoidal and need not be a REF.
:::


:::{note} **A Deeper Look: The Rank-One Decomposition View**
:label: note-rank-one-matrix-decomposition

If we interpret the matrix product $\mathbf{LU}$ as a **linear combination of rank-one matrices**, we discover an interesting structure:

$$
\mathbf{LU} = \underbrace{\begin{bmatrix} \textcolor{#FF8C00}{1} \\ \textcolor{#FF8C00}{2} \\ \textcolor{#FF8C00}{-1} \end{bmatrix} \begin{bmatrix} \textcolor{#FF8C00}{2} & \textcolor{#FF8C00}{1} & \textcolor{#FF8C00}{1} \end{bmatrix}}_{\text{rank-one matrix 0 (3×3)}} + \underbrace{\begin{bmatrix} 0 \\ \textcolor{#006385}{1} \\ \textcolor{#006385}{-1} \end{bmatrix} \begin{bmatrix} 0 & \textcolor{#006385}{-8} & \textcolor{#006385}{-2} \end{bmatrix}}_{\text{rank-one matrix 1 (effectively 2×2)}} + \underbrace{\begin{bmatrix} 0 \\ 0 \\ \textcolor{#57068C}{1} \end{bmatrix} \begin{bmatrix} 0 & 0 & \textcolor{#57068C}{1} \end{bmatrix}}_{\text{rank-one matrix 2 (effectively 1×1)}}
$$

Looking at this decomposition, we find:
- $\textcolor{#FF8C00}{\square}$ **The orange part**: corresponds to the coefficients of round 0 of the elimination and forms an effective $3 \times 3$ region
- $\textcolor{#006385}{\square}$ **The blue part**: corresponds to the coefficients of round 1 of the elimination and forms an effective $2 \times 2$ region
- $\textcolor{#57068C}{\square}$ **The violet part**: corresponds to the coefficients of round 2 of the elimination and forms an effective $1 \times 1$ region

This pattern tells us that when constructing the matrix $\mathbf{L}$, **we do not actually need to multiply all the elementary row-operation matrices together**; we only need to enter the multipliers of each round of elimination directly into the corresponding positions! This simplifies the computation enormously.
:::

### 6.3.3 Python/Simplified LU Decomposition

The following code implements the algorithm above; please read it alongside the steps.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ──────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

# ──────────────────────────────────────────────
def simplified_lu_decomposition(A, tol=1e-12):
    """
    Simplified LU decomposition (no row exchanges)

    Parameters
    ----------
    A   : input matrix, of shape m×n
    tol : numerical tolerance for deciding whether a pivot is zero

    Returns
    -------
    L       : lower triangular matrix, of shape m×m
    U       : upper triangular matrix, of shape m×n
    success : Boolean indicating whether the decomposition succeeded
    """
    m, n = A.shape
    L = np.eye(m)                    # initialize L as the m×m identity matrix
    U = A.copy().astype(float)       # initialize U as a floating-point copy of A

    for k in range(min(m - 1, n)):

        # --- Step 1: check the pivot U[k, k] ---
        if abs(U[k, k]) >= tol:
            pass  # the pivot is usable; go straight to elimination

        else:
            # --- Step 2: the pivot is close to zero; check below it in this column ---
            column_below = U[k + 1:m, k]
            if np.any(np.abs(column_below) >= tol):
                # there is a nonzero entry below, but row exchanges are not allowed
                print(f"  [!] The pivot in row {k}, column {k} is zero, and there is a nonzero entry below it")
                print(f"      The simplified LU decomposition does not support row exchanges; the decomposition fails")
                return L, U, False
            else:
                # the whole column is close to zero; skip this column
                print(f"  [i] The column from row {k}, column {k} down is close to zero; skipping")
                continue

        # --- Step 3: eliminate in column k ---
        for i in range(k + 1, m):
            if abs(U[i, k]) >= tol:
                multiplier = U[i, k] / U[k, k]   # elimination multiplier l_{ik}
                L[i, k]    = multiplier            # store in L
                U[i, k:n]  = U[i, k:n] - multiplier * U[k, k:n]
                U[i, k]    = 0.0                   # set to zero explicitly to remove floating-point residue

    return L, U, True

# ============================================================
print_header("Example 6.2 | Simplified LU Decomposition")
# ============================================================

# --- Step 1: define the matrix A ---
print_step(1, "Define the matrix A")
A = np.array([
    [ 2.0,  1.0,  1.0],
    [ 4.0, -6.0,  0.0],
    [-2.0,  7.0,  2.0]
], dtype=float)
print("A =\n", A)

# --- Step 2: perform the simplified LU decomposition ---
print_step(2, "Perform the simplified LU decomposition (no row exchanges)")
L, U, success = simplified_lu_decomposition(A)

if not success:
    print("\n  The decomposition failed; use the PLU decomposition with row exchanges instead.")
else:
    print("  Decomposition succeeded ✓")

    # --- Step 3: print L and U ---
    print_step(3, "Result of the decomposition: L and U")
    print("L =\n", L)
    print("\nU =\n", U)

    # --- Step 4: check L @ U = A ---
    print_step(4, "Check L·U = A")
    LU = L @ U
    error = np.max(np.abs(A - LU))
    compare_print("L · U", LU, "should recover the original matrix A")
    compare_print("Maximum reconstruction error ‖A − LU‖∞", f"{error:.2e}", "should be close to machine precision ~1e-16")

    # --- Step 5: check the upper triangular structure of U (all zeros below the diagonal) ---
    print_step(5, "Check that U is upper triangular: U[i, j] = 0 when i > j")
    below_diag = np.max(np.abs(np.tril(U, k=-1)))
    compare_print("Largest absolute value below the diagonal max|U[i, j]| (i > j)", f"{below_diag:.2e}", "should be 0")
    assert error < 1e-10, "Reconstruction error too large: L·U does not recover A"
    assert below_diag < 1e-12, "U is not upper triangular: nonzero entries remain below the diagonal"

# --- Step 6: test with a tall matrix (m > n): the last elimination position (column 1) must not be skipped ---
print_step(6, "Tall-matrix test: the 3×2 matrix A_tall")
A_tall = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 2.0]
])
L_t, U_t, ok_t = simplified_lu_decomposition(A_tall)
print("U_tall =\n", U_t)
err_t   = np.max(np.abs(A_tall - L_t @ U_t))
below_t = np.max(np.abs(np.tril(U_t, k=-1)))
compare_print("‖A_tall − LU‖∞ / largest value below the diagonal of U", f"{err_t:.2e} / {below_t:.2e}", "both should be 0")
assert ok_t and err_t < 1e-10 and below_t < 1e-12, "The LU decomposition of a tall matrix should both reconstruct A and be upper triangular"


#### Why Is the LU Decomposition Written $\mathbf{A} = \mathbf{L}\mathbf{U}$ Rather Than $\mathbf{L}\mathbf{A} = \mathbf{U}$?

Each step of Gaussian elimination **left-multiplies** $\mathbf{A}$ by an **elementary lower triangular matrix** $\mathbf{E}_k$, clearing the entries below the pivot in column $k$:

$$
\mathbf{E}_{n-1} \cdots \mathbf{E}_1 \mathbf{E}_0 \,\mathbf{A} = \mathbf{U}
$$

So from the viewpoint of the elimination operations, the notation $\mathbf{L}\mathbf{A} = \mathbf{U}$ is not entirely wrong—it describes "applying a sequence of row operations to $\mathbf{A}$ to obtain $\mathbf{U}$."

But there are three fundamental reasons for writing it in the end as $\mathbf{A} = \mathbf{L}\mathbf{U}$:

**Reason 1: the inverses of the $\mathbf{E}_k$ have a more elegant structure**

Each $\mathbf{E}_k$ is a unit lower triangular matrix, and its inverse $\mathbf{E}_k^{-1}$ is obtained simply by **changing the signs** of the elimination multipliers, with exactly the same structure. Let

$$
\mathbf{L} = \mathbf{E}_0^{-1} \mathbf{E}_1^{-1} \cdots \mathbf{E}_{n-1}^{-1}
$$

Then $\mathbf{L}$ is precisely the lower triangular matrix obtained by entering all the elimination multipliers directly into their corresponding positions, with no extra computation. This is exactly what the code below demonstrates: multiplying the $\mathbf{E}_k^{-1}$ **in order of increasing index** ($\mathbf{E}_0^{-1}\mathbf{E}_1^{-1}\cdots\mathbf{E}_{n-1}^{-1}$, that is, in the **reverse** of the order in which the eliminations are performed) gives a product that is exactly the $\mathbf{L}$ assembled directly from the multipliers of each round; if they are multiplied in order of decreasing index ($\mathbf{E}_{n-1}^{-1}\cdots\mathbf{E}_0^{-1}$), the product is still a unit lower triangular matrix but is **not equal** to $\mathbf{L}$.

**Reason 2: $\mathbf{A} = \mathbf{L}\mathbf{U}$ is a decomposition of $\mathbf{A}$, while $\mathbf{L}\mathbf{A} = \mathbf{U}$ is a transformation of $\mathbf{A}$**

The meaning of a decomposition is "to express $\mathbf{A}$ as the product of two matrices of simple structure," for use in solving, analysis, and storage. The meaning of $\mathbf{L}\mathbf{A} = \mathbf{U}$, by contrast, is "to apply operations to $\mathbf{A}$"; it describes a process rather than a structure.

**Reason 3: the applications are built around $\mathbf{A} = \mathbf{L}\mathbf{U}$**

When solving the linear system $\mathbf{A}\mathbf{x} = \mathbf{b}$, the decomposition gives

$$
\mathbf{L}\mathbf{U}\mathbf{x} = \mathbf{b} \implies \mathbf{L}\mathbf{y} = \mathbf{b},\quad \mathbf{U}\mathbf{x} = \mathbf{y}
$$

Both substitution steps use $\mathbf{L}$ and $\mathbf{U}$ directly rather than the sequence of $\mathbf{E}_k$. The form $\mathbf{A} = \mathbf{L}\mathbf{U}$ makes this procedure natural and symmetric.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ──────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value:\n{actual}")
    print(f"  Expected value: {expected_desc}")

# ============================================================
print_header("Example 6.3 | The Order of the Product of the Inverse Elementary Matrices E_k⁻¹ and L")
# ============================================================

n = 4   # size for the demonstration; adjust as you like

np.random.seed(42)   # fix the random seed so the results are reproducible

# --- Step 1: generate n unit lower triangular matrices E_0⁻¹, …, E_{n-1}⁻¹ ---
# Note: the elimination matrix E_k in the text places the multipliers, "with their signs changed," below the pivot in column k;
# its inverse E_k⁻¹ places the elimination multipliers "unchanged (with positive sign)" in the same positions.
# Here we directly generate the E_k⁻¹ of each round (with positive multipliers), matching L = E_0⁻¹E_1⁻¹⋯E_{n-1}⁻¹.
print_step(1, f"Generate {n} unit lower triangular matrices E_0⁻¹, …, E_{{n-1}}⁻¹ (each holding the multipliers of round k)")

Einv_list = []
for k in range(n):
    Einv = np.eye(n)
    num_fill = n - k - 1          # number of multipliers to fill in below the pivot in column k
    if num_fill > 0:
        Einv[k+1:, k] = np.round(np.random.uniform(1, 10, num_fill), 2)
    Einv_list.append(Einv)
    print(f"\nE_{k}⁻¹ =\n", Einv)

# --- Step 2: multiply in order  E_0⁻¹ @ E_1⁻¹ @ … @ E_{n-1}⁻¹ (increasing index) ---
print_step(2, "Multiply in order: E_0⁻¹ · E_1⁻¹ · … · E_{n-1}⁻¹ (increasing index)")

forward = Einv_list[0].copy()
for Einv in Einv_list[1:]:
    forward = forward @ Einv
print("Product in order =\n", forward)
print("\n  ➜ The product in order is a unit lower triangular matrix and is exactly the L of the LU decomposition (see the check in Step 4)")

# --- Step 3: multiply in reverse order  E_{n-1}⁻¹ @ … @ E_1⁻¹ @ E_0⁻¹ (decreasing index) ---
print_step(3, "Multiply in reverse order: E_{n-1}⁻¹ · … · E_1⁻¹ · E_0⁻¹ (decreasing index)")

reverse = Einv_list[-1].copy()
for Einv in reversed(Einv_list[:-1]):
    reverse = reverse @ Einv
print("Product in reverse order =\n", reverse)
print("\n  ➜ The product in reverse order is still a unit lower triangular matrix, but it is not L (see the check in Step 4)")

# --- Step 4: check: product in order = L (assembled directly), product in reverse order ≠ L ---
print_step(4, "Check: product in order = the L assembled directly from the multipliers of each column; product in reverse order ≠ L")

L_direct = np.eye(n)
for k, Einv in enumerate(Einv_list):
    L_direct[k+1:, k] = Einv[k+1:, k]   # enter the elimination multipliers of each E_k⁻¹ directly

print("\nL assembled directly =\n", L_direct)

compare_print(
    "Maximum difference ‖product in order − L_direct‖∞",
    f"{np.max(np.abs(forward - L_direct)):.2e}",
    "should be 0 (or machine precision): the product in order is L"
)
compare_print(
    "Maximum difference ‖product in reverse order − L_direct‖∞",
    f"{np.max(np.abs(reverse - L_direct)):.2e}",
    "should be clearly nonzero: in the product in reverse order the multipliers contaminate one another, so it is not L"
)
assert np.allclose(forward, L_direct), "The product in order should equal the directly assembled L"
assert not np.allclose(reverse, L_direct), "The product in reverse order should not equal L"
print("\n  ✓ Conclusion: L = E_0⁻¹E_1⁻¹⋯E_{n-1}⁻¹ (in order of increasing index); the product in reverse order is not.")


### 6.3.4 Complete Gaussian Elimination and the PLU Decomposition

As described in the previous subsection, the ideal LU decomposition is often not feasible in actual computation, because:
- a diagonal entry may be zero and so cannot serve as the pivot entry
- even when it is nonzero, a very small value leads to numerical instability

Practical Gaussian elimination therefore needs to introduce a **row-exchange strategy** (in engineering practice, column exchanges are also added to improve the stability of the algorithm further), which gives the **PLU decomposition**.

**Permutation matrices and row exchanges**

A permutation matrix is a matrix obtained by rearranging the rows (or the columns) of the identity matrix. Left-multiplying a matrix by a permutation matrix changes the order of the rows of that matrix.

:::{prf:definition} Mathematical Properties of Permutation Matrices
:label: def-permutation-matrix
A **permutation matrix** $\mathbf{P}$ is a matrix obtained by permuting the rows of the identity matrix. Mathematically, $\mathbf{P}$ satisfies:
1. Each row and each column has exactly one entry equal to 1, and all other entries are 0
2. $\mathbf{P}\mathbf{P}^\top = \mathbf{P}^\top\mathbf{P} = \mathbf{I}$
:::

An important property of permutation matrices is that their inverse equals their transpose: $\mathbf{P}^{-1} = \mathbf{P}^\top$.


:::{prf:algorithm} Complete Gaussian Elimination with Row Exchanges: The PLU Decomposition
:label: alg-flexible-plu

**Step 1. Preparation**
- A numerical implementation sets an absolute threshold (say 1e-12); the result is an approximate echelon form and a numerical number of pivots at this tolerance, which depend on the scale of the data and cannot be equated with the exact rank. The exact theorems decide by whether an entry equals zero. If a numerical rank that is robust to scale is needed, one should instead use a relative tolerance (for example, the largest absolute value of the entries of the matrix as the scale, multiplied by the tolerance), together with a rank-revealing decomposition (such as the SVD of Chapter 11)
- Prepare three matrices: set the matrix P to the identity matrix (to record the row exchanges), set the matrix L to the identity matrix, and let the matrix U be a copy of the original matrix A
- Set two indices: `current_row = 0` (the row currently being processed) and `current_col = 0` (the column currently being searched for a pivot)

**Step 2. Search flexibly for a pivot and process it**
Keep processing while `current_row < m` and `current_col < n`:

> **For the current position (current_row, current_col):**
> 
> 1. **Find the best pivot in the current column**
>    - In column `current_col`, search downward starting from `current_row`
>    - Take the entry of largest absolute value as the candidate pivot
>    - Record the row index `best_row` of that entry
> 
> 2. **Check whether the pivot is usable**
>    - If the largest entry is still below the threshold:
>      - the whole column (from current_row on) is close to zero
>      - **move on to the next column**: `current_col += 1`
>      - go back to step 1 and keep searching
>    - If a usable pivot is found, go to step 3
> 
> 3. **Perform a row exchange (if needed)**
>    - If `best_row != current_row`:
>      - interchange row `current_row` and row `best_row` of the matrix U
>      - update the matrix P to record this exchange
>      - interchange the corresponding entries of the already computed part of the matrix L
> 
> 4. **Perform the elimination**
>    - Pivot position: `(current_row, current_col)`
>    - Pivot value: `pivot = U[current_row, current_col]`
>    - For each row `i` below `current_row`:
>      - compute the elimination multiple: `multiplier = U[i, current_col] / pivot`
>      - record it in the matrix L: `L[i, current_row] = multiplier`
>      - eliminate: `U[i, :] -= multiplier * U[current_row, :]`
>      - clear to zero: `U[i, current_col] = 0`
> 
> 5. **Update the position being processed**
>    - `current_row += 1`
>    - `current_col += 1`
>    - continue with the next round

**Step 3. Finish and verify**
- We obtain three matrices: $\mathbf{P}$ (recording the row exchanges), $\mathbf{L}$ (lower triangular), and $\mathbf{U}$ (in row echelon form, REF)
- The pivots of the matrix $\mathbf{U}$ may not lie on the diagonal; in exact arithmetic it is in REF, while the implementation with a tolerance yields only an approximate REF
- Verify: $$\mathbf{PA=LU}$$ or $\mathbf{A}=\mathbf{P}^{\top}\mathbf{LU}$

**Key improvements over the LU decomposition.**
1. **Flexible pivot search**: not restricted to the diagonal positions (k,k)
2. **Skipping zero columns**: when a column has no usable pivot, the algorithm automatically moves on to the next column
3. **Adapting to the structure of the matrix**: any m×n matrix can be handled
4. **Pivot positions**: the column indices of the pivots are strictly increasing; in the version with a tolerance, the skipped entries are only close to zero

:::


The early stages of the PLU decomposition are no different from the LU decomposition, until during the elimination we find that rows $k$ and $i$ ($i > k$) need to be interchanged. Because row exchanges and row eliminations are interleaved, the key question is: **how can the PLU form of the decomposition be preserved while they are interleaved?** 

**Block representation of the current state.**

Suppose elimination has been completed for the first $k$ rows; the state of the matrix can then be written as
$$\mathbf{P}_k\mathbf{A} = \mathbf{L}_k \mathbf{U}_k = \begin{bmatrix} 
\mathbf{L}_{00} & \mathbf{0} \\ 
\mathbf{L}_{10} & \mathbf{I} 
\end{bmatrix} \begin{bmatrix} 
\mathbf{U}_{00} & \mathbf{U}_{01} \\ 
\mathbf{0} & \mathbf{U}_{11} 
\end{bmatrix}$$

where:
- $\mathbf{P}_k$: the permutation matrix accumulated over the first $k$ steps
- $\mathbf{L}_{00}$: the completed $k \times k$ unit lower triangular block; $\mathbf{L}_{10}$: the completed $(m-k) \times k$ matrix block
- $\mathbf{U}_{00}$: the completed $k \times s$ generalized upper triangular block, also in REF form, with $k \leq s$
- $\mathbf{U}_{11}$: the $(m-k) \times (n-s)$ block still to be processed

**The row exchange operation.**

When rows $k$ and $i$ need to be interchanged, we construct a permutation matrix acting only on the trailing rows:
$$\mathbf{Q}_k = \begin{bmatrix} 
\mathbf{I}_k & \mathbf{0} \\ 
\mathbf{0} & \mathbf{Q}'_{ki} 
\end{bmatrix}$$

where $\mathbf{Q}'_{ki}$ is an $(m-k) \times (m-k)$ permutation matrix responsible for interchanging the corresponding rows.

**Key lemma.** Row exchanges do not destroy the lower triangular structure of $\mathbf{L}$

This is the heart of why the PLU decomposition holds in general. We first check it on a $3 \times 3$ block example, and then explain the general case.

**A worked check.** Let $k=1$ (step 0 of the elimination is complete), and interchange row 1 and row 2. Then

$$
\mathbf{L}_1 = \begin{bmatrix} 1 & 0 & 0 \\ m_{10} & 1 & 0 \\ m_{20} & 0 & 1 \end{bmatrix}, \qquad
\mathbf{Q}_1 = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 0 & 1 \\ 0 & 1 & 0 \end{bmatrix}
$$

Compute $\mathbf{Q}_1 \mathbf{L}_1 \mathbf{Q}_1$ directly:

**Step 1.** First compute $\mathbf{L}_1 \mathbf{Q}_1$ (right multiplication by the permutation = interchanging **columns** 1 and 2 of $\mathbf{L}_1$):

$$
\mathbf{L}_1 \mathbf{Q}_1 = \begin{bmatrix} 1 & 0 & 0 \\ m_{10} & 0 & 1 \\ m_{20} & 1 & 0 \end{bmatrix}
$$

**Step 2.** Then compute $\mathbf{Q}_1 (\mathbf{L}_1 \mathbf{Q}_1)$ (left multiplication by the permutation = interchanging **rows** 1 and 2):

$$
\mathbf{Q}_1 \mathbf{L}_1 \mathbf{Q}_1 = \begin{bmatrix} 1 & 0 & 0 \\ m_{20} & 1 & 0 \\ m_{10} & 0 & 1 \end{bmatrix}
$$

The result is still a unit lower triangular matrix—only the positions of $m_{10}$ and $m_{20}$ have been swapped (corresponding to the exchange of row 1 and row 2).

**The general case.** The structure of $\mathbf{Q}_k$ guarantees that it permutes only the row indices after step $k$ and does not touch the completed $k \times k$ upper-left block $\mathbf{L}_{00}$; and the lower-right block is $\mathbf{I}$ before the row exchange and is still $\mathbf{I}$ afterward (rows and columns are interchanged together, which leaves the identity matrix unchanged). Therefore

$$\mathbf{Q}_k \mathbf{L}_k \mathbf{Q}_k = \begin{bmatrix} 
\mathbf{L}_{00} & \mathbf{0} \\ 
\mathbf{Q}' \mathbf{L}_{10} & \mathbf{I} 
\end{bmatrix}$$

where $\mathbf{Q}' \mathbf{L}_{10}$ is only a rearrangement of the multipliers already recorded and introduces no entries outside the lower triangle.

$\mathbf{L}'_k = \mathbf{Q}_k \mathbf{L}_k \mathbf{Q}_k$ **is still a unit lower triangular matrix**.

**Updating the decomposition.**

Using the property $\mathbf{Q}_k \mathbf{Q}_k = \mathbf{I}$ of the permutation matrix, we obtain

$$ (\mathbf{Q}_k\mathbf{P}_k)\mathbf{A}=(\mathbf{Q}_k\mathbf{L}_k\mathbf{Q}_k)(\mathbf{Q}_k\mathbf{U}_k)=\mathbf{L}'_k\mathbf{U}'_k. $$

where:
- $\mathbf{P}_{k+1} = \mathbf{Q}_k \mathbf{P}_k$: the updated accumulated permutation matrix
- $\mathbf{L}'_k = \mathbf{Q}_k \mathbf{L}_k \mathbf{Q}_k$: the updated unit lower triangular matrix
- $\mathbf{U}'_k = \mathbf{Q}_k \mathbf{U}_k$: the partially upper triangular matrix after the row exchange, which is also partially in REF

**Key insight.** By this clever device we convert "row exchanges during the computation" into "row exchanges before the computation begins," always maintaining the standard form $\mathbf{PA} = \mathbf{LU}$. In other words, if we knew the final permutation matrix $\mathbf{P}$ in advance, the ideal LU decomposition of $\mathbf{PA}$ would need no row exchanges at all!

:::{prf:remark} Block Elimination and the Schur Complement
:label: rem-block-elimination-schur

The block language of this section is in fact only one step away from the Schur complement of Chapter 5. For a $2 \times 2$ block matrix $\mathbf{M} = \begin{bmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{bmatrix}$ (with $\mathbf{A}$ invertible), perform "one step of **block elimination**"—use the upper-left block to eliminate the lower-left block, that is, left-multiply by a block unit lower triangular matrix:

$$
\begin{bmatrix} \mathbf{I} & \mathbf{0} \\ -\mathbf{C}\mathbf{A}^{-1} & \mathbf{I} \end{bmatrix}
\begin{bmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{bmatrix}
= \begin{bmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{0} & \mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B} \end{bmatrix}
$$

The $\mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B}$ that appears in the lower-right corner is exactly the **Schur complement** in the block inversion formula of Chapter 5, §5.3.2: what it really is, is "the lower-right block left to be processed after one round of Gaussian elimination on the blocks." When Chapter 7 computes the determinant of a block matrix, this observation will directly give $\det \mathbf{M} = \det \mathbf{A} \cdot \det(\mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B})$.
:::


:::{prf:theorem} Existence of the PLU Decomposition
:label: thm-plu-existence
Let $\mathbf{A}\in\mathbb{R}^{m\times n}$. Then there exist an $m\times m$ permutation matrix $\mathbf{P}$,
an $m\times m$ unit lower triangular matrix $\mathbf{L}$, and an $m\times n$ row echelon matrix $\mathbf{U}$ such that
$$\mathbf{PA}=\mathbf{LU}.$$
In this case $\operatorname{rank}(\mathbf{A})=\operatorname{rank}(\mathbf{U})$,
which is the number of nonzero rows of $\mathbf{U}$.
:::

:::{prf:proof}
Algorithm {prf:ref}`alg-flexible-plu` gives a constructive procedure: in each round of elimination, if the current column contains a nonzero entry, choose the one of largest absolute value as the pivot and perform the row exchange and the elimination; if the whole column is zero, move on to the next column. This procedure necessarily terminates after finitely many steps (each step advances the row index or the column index by at least one), and at termination it yields $\mathbf{P}$, $\mathbf{L}$, $\mathbf{U}$.

This proof uses exact arithmetic and decides by equality with zero; the version with a tolerance in the program is only an approximate implementation. After the elimination matrix $\mathbf{E}_k$ eliminates the entries below the pivot, the updates $\mathbf{U}_{k+1}=\mathbf{E}_k\mathbf{U}^{\prime}_k$ and $\mathbf{L}_{k+1}=\mathbf{L}^{\prime}_k\mathbf{E}_k^{-1}$ preserve the decomposition and the unit lower triangular structure. It remains to verify that the interleaved row exchanges do not destroy the lower triangular structure of $\mathbf{L}$. By the block derivation above: when step $k$ requires a row exchange, construct the permutation matrix $\mathbf{Q}_k$ acting only on the trailing rows and insert it using $\mathbf{Q}_k^2 = \mathbf{I}$:

$$ (\mathbf{Q}_k\mathbf{P}_k)\mathbf{A}=(\mathbf{Q}_k\mathbf{L}_k\mathbf{Q}_k)(\mathbf{Q}_k\mathbf{U}_k). $$

Here $\mathbf{Q}_k \mathbf{L}_k \mathbf{Q}_k$ only permutes the row indices of the completed part of $\mathbf{L}_k$, so **the unit lower triangular structure is preserved**. After the permutations are accumulated step by step as $\mathbf{P}_{k+1} = \mathbf{Q}_k \mathbf{P}_k$, the whole process maintains the form $\mathbf{PA} = \mathbf{L}\mathbf{U}$ until it terminates.

$\underline{\text{For every }m \times n\text{ matrix, a PLU decomposition exists.}}$ $\square$
:::

**Practical significance**

The advantages of the PLU decomposition are:

1. **Numerical stability**: choosing the pivot entry of largest absolute value greatly reduces the accumulation of rounding errors
2. **General applicability**: it applies to every matrix and is not restricted by singularity  
3. **Computational efficiency**: although row exchange operations are added, the overall complexity is still $O(mn \min(m,n))$
4. **Theoretical completeness**: it provides a reliable numerical foundation for the many applications of linear algebra

In practice, the PLU decomposition is the core tool for solving systems of linear equations, computing determinants, finding inverse matrices, and similar operations.

:::{prf:example} An Example of the PLU Decomposition of a 4×4 Matrix
:label: ex-plu-4x4

Find the PLU decomposition of the matrix

$$\mathbf{A} = \begin{bmatrix}
1 & 1 & 0 & 0 \\
1/2 & 1/2 & 1 & 0 \\
1/3 & 4/3 & 1 & 0\\
1/4 & 1/4 & 1 & 1
\end{bmatrix}$$

where:
- $\text{row}_0(\mathbf{A}) = [1, 1, 0, 0]$
- $\text{row}_1(\mathbf{A}) = [1/2, 1/2, 1, 0]$
- $\text{row}_2(\mathbf{A}) = [1/3, 4/3, 1, 0]$
- $\text{row}_3(\mathbf{A}) = [1/4, 1/4, 1, 1]$

**Step 1. Elimination with row 0**

Look at the entries of column 0: $[1, 1/2, 1/3, 1/4]$
The largest absolute value is 1 (in $\text{row}_0$), so no row exchange is needed.

Use $\text{row}_0$ to eliminate the entries below:
- $\text{row}_1 - \frac{1}{2}\text{row}_0$: $[1/2, 1/2, 1, 0] - \frac{1}{2}[1, 1, 0, 0] = [0, 0, 1, 0]$
- $\text{row}_2 - \frac{1}{3}\text{row}_0$: $[1/3, 4/3, 1, 0] - \frac{1}{3}[1, 1, 0, 0] = [0, 1, 1, 0]$
- $\text{row}_3 - \frac{1}{4}\text{row}_0$: $[1/4, 1/4, 1, 1] - \frac{1}{4}[1, 1, 0, 0] = [0, 0, 1, 1]$

This gives

$$\begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 1 \end{bmatrix}$$

**Step 2. Elimination with row 1**

Look at the entries of column 1 starting from row 1: $[0, 1, 0]$ (from $\text{row}_1, \text{row}_2, \text{row}_3$)
The largest absolute value is 1 (in $\text{row}_2$), so $\text{row}_1$ and $\text{row}_2$ must be interchanged.

After the exchange we have

$$\begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 1 & 1 \end{bmatrix}$$

The entries of column 1 below $\text{row}_1$ are already 0, so no further elimination is needed.

**Step 3. Elimination with row 2**

Look at the entries of column 2 starting from row 2: $[1, 1]$ (from $\text{row}_2, \text{row}_3$)
Choose $\text{row}_2$ as the pivot row.

Eliminate: $\text{row}_3 - 1 \cdot \text{row}_2 = [0, 0, 1, 1] - [0, 0, 1, 0] = [0, 0, 0, 1]$

The final upper triangular matrix is

$$\mathbf{U} = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

**The result of the PLU decomposition**

**The permutation matrix P** (recording the row exchange: $\text{row}_1$ and $\text{row}_2$ interchanged):

$$\mathbf{P} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

**The lower triangular matrix L** (recording the elimination multipliers, with the row exchange taken into account):

$$\mathbf{L} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 1/3 & 1 & 0 & 0 \\ 1/2 & 0 & 1 & 0 \\ 1/4 & 0 & 1 & 1 \end{bmatrix}$$

**The upper triangular matrix U**:

$$\mathbf{U} = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

**Verification**

One can verify that $\mathbf{PA} = \mathbf{LU}$:

$$\mathbf{PA} = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 1/3 & 4/3 & 1 & 0 \\ 1/2 & 1/2 & 1 & 0 \\ 1/4 & 1/4 & 1 & 1 \end{bmatrix} = \mathbf{LU}$$

Hence the PLU decomposition of the matrix A is $\mathbf{A} = \mathbf{P}^{-1}\mathbf{LU}$, where $\mathbf{P}^{-1} = \mathbf{P}^\top$.

**Key insight.** The heart of the PLU decomposition lies in understanding the role of the row exchanges. When we perform row exchanges, we are in effect carrying out an LU decomposition, with no row exchanges, of $\mathbf{PA}$ (the matrix with its rows exchanged in advance). Therefore the elimination multipliers in the matrix L reflect the coefficients of the elimination performed on $\mathbf{PA}$, not a direct record of the elimination process on the original matrix $\mathbf{A}$.
:::

### 6.3.5 Python/Complete PLU Decomposition

The following code implements the algorithm above; please read it alongside the steps

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ──────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

# ──────────────────────────────────────────────
def flexible_plu_decomposition(A, tol=1e-12):
    """
    PLU decomposition with flexible pivoting (pivots not restricted to diagonal positions)

    Parameters
    ----------
    A   : input matrix, of shape m×n
    tol : numerical tolerance for deciding whether a pivot is zero

    Returns
    -------
    P               : permutation matrix, of shape m×m
    L               : unit lower triangular matrix, of shape m×m
    U               : approximate REF matrix at this tolerance (a REF in exact arithmetic), of shape m×n
    pivot_positions : list of pivot positions [(row, col), ...]
    row_swaps       : record of row exchanges [(row_i, row_j), ...]
    success         : Boolean indicating whether the decomposition succeeded
    """
    m, n = A.shape
    P = np.eye(m)
    L = np.eye(m)
    U = A.copy().astype(float)

    row_swaps       = []
    pivot_positions = []
    current_row     = 0
    current_col     = 0

    while current_row < m and current_col < n:

        # --- Step 1: find the pivot of largest absolute value in the current column ---
        col_segment      = U[current_row:m, current_col]
        rel_best         = np.argmax(np.abs(col_segment))
        best_row         = current_row + rel_best
        best_val         = U[best_row, current_col]

        # --- Step 2: if the pivot is too small, skip this column ---
        if abs(best_val) < tol:
            current_col += 1
            continue

        # --- Step 3: if the pivot is not in the current row, perform a row exchange ---
        if best_row != current_row:
            row_swaps.append((current_row, best_row))
            U[[current_row, best_row]]    = U[[best_row, current_row]]
            P[[current_row, best_row]]    = P[[best_row, current_row]]
            if current_row > 0:
                L[[current_row, best_row], :current_row] = \
                    L[[best_row, current_row], :current_row]

        pivot_positions.append((current_row, current_col))
        pivot = U[current_row, current_col]

        # --- Step 4: eliminate, clearing the entries below the pivot ---
        for i in range(current_row + 1, m):
            if abs(U[i, current_col]) >= tol:
                mult             = U[i, current_col] / pivot
                L[i, current_row] = mult
                U[i, current_col:] = U[i, current_col:] - mult * U[current_row, current_col:]
                U[i, current_col]  = 0.0

        current_row += 1
        current_col += 1

    return P, L, U, pivot_positions, row_swaps, True


# ──────────────────────────────────────────────
def verify_flexible_plu(A, P, L, U, tol=1e-10):
    """
    Verify the PLU decomposition: PA = LU, equivalently A = P^T·LU

    Returns bool: whether the verification passes
    """
    err_PA_LU = np.max(np.abs(P @ A - L @ U))
    err_A_rec = np.max(np.abs(A - P.T @ L @ U))
    print(f"  ‖PA − LU‖∞  = {err_PA_LU:.2e}")
    print(f"  ‖A − P^T LU‖∞ = {err_A_rec:.2e}")
    return max(err_PA_LU, err_A_rec) < tol


def analyze_flexible_plu(P, L, U, pivot_positions, row_swaps):
    """
    Analyze the result of the PLU decomposition: number of row exchanges, pivot positions, rank of the matrix
    """
    m, n = U.shape
    print(f"  Number of row exchanges: {len(row_swaps)}")
    if row_swaps:
        for k, (r1, r2) in enumerate(row_swaps):
            print(f"    Exchange {k+1}: row {r1} ↔ row {r2}")
    print(f"  Numerical rank at this tolerance (number of pivots): {len(pivot_positions)}")
    for k, (r, c) in enumerate(pivot_positions):
        print(f"    Pivot {k}: position ({r}, {c}), value = {U[r, c]:.4f}")
    pivot_cols = [c for _, c in pivot_positions]
    if len(pivot_cols) > 1:
        ok = all(pivot_cols[i] < pivot_cols[i+1] for i in range(len(pivot_cols)-1))
        print(f"  Pivot column positions strictly increasing (REF property): {ok}")


# ──────────────────────────────────────────────
def run_case(label, A):
    """Run the complete procedure for a single test case"""
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print("A =\n", A)

    P, L, U, positions, swaps, _ = flexible_plu_decomposition(A)

    print("\nP =\n", P)
    print("\nL =\n", L)
    print("\nU =\n", U)

    print("\n  [Verification]")
    passed = verify_flexible_plu(A, P, L, U)

    print("\n  [Analysis]")
    analyze_flexible_plu(P, L, U, positions, swaps)

    compare_print(
        "Verification result",
        "passed ✓" if passed else "failed ✗",
        "PA = LU and A = P^T LU; the errors should be close to machine precision"
    )


# ============================================================
print_header("Example 6.4 | PLU Decomposition with Flexible Pivoting")
# ============================================================

# --- Step 1: a square matrix that needs a row exchange ---
print_step(1, "A square matrix A₁ that needs a row exchange")
A1 = np.array([
    [1.0,     1.0,     0.0, 0.0],
    [1.0/2,   1.0/2,   1.0, 0.0],
    [1.0/3,   4.0/3,   1.0, 0.0],
    [1.0/4,   1.0/4,   1.0, 1.0],
])
run_case("Case 1: a square matrix that needs a row exchange", A1)

# --- Step 2: a matrix whose pivots are not on the diagonal ---
print_step(2, "A matrix A₂ whose pivots move off the diagonal after elimination")
A2 = np.array([
    [1.0, 2.0, 0.0, 3.0],
    [2.0, 4.0, 1.0, 7.0],
    [1.0, 2.0, 2.0, 8.0],
])
run_case("Case 2: pivots off the diagonal (column 1 becomes a zero column after elimination)", A2)

# --- Step 3: a rectangular matrix (number of rows < number of columns) ---
print_step(3, "A rectangular matrix A₃ (3×4)")
A3 = np.array([
    [1.0, 2.0, 3.0, 4.0],
    [2.0, 1.0, 4.0, 3.0],
    [1.0, 3.0, 2.0, 5.0],
])
run_case("Case 3: a rectangular matrix", A3)

# --- Step 4: a tiny pivot, which needs a row exchange to maintain numerical stability ---
print_step(4, "A matrix A₄ with a tiny pivot (demonstrating numerical stability)")
A4 = np.array([
    [1e-10, 1.0, 2.0],
    [1.0,   2.0, 1.0],
    [2.0,   1.0, 3.0],
])
run_case("Case 4: tiny pivot → a row exchange ensures numerical stability", A4)

### 6.3.6 A Unified Framework for Solving Linear Systems and Inverting Matrices

The PLU decomposition obtained in the previous subsection can not only solve the system of linear equations $\mathbf{Ax = b}$ but also compute the inverse of a matrix efficiently. In fact, **finding the inverse matrix is equivalent to solving n special systems of linear equations simultaneously**, $\mathbf{Ax_i = e_i}$ (where $\mathbf{e_i}$ is the i-th column vector of the identity matrix), so that Gaussian elimination, the LU decomposition, and the PLU decomposition provide a unified theoretical and computational framework for solving systems of linear equations and computing inverse matrices.

**Features of Gaussian elimination.** In essence, it transforms the system of linear equations $\mathbf{Ax = b}$ by a sequence of row operations into an equivalent upper triangular system $\mathbf{Ux = b\prime}$. The "advantage" of Gaussian elimination is that only the final simplified system needs to be recorded, not the transformation process, which makes it look "simple and intuitive"; even high school students can master it. However, because the transformation process is not recorded, every elimination step must be carried out again each time a new constant vector $\mathbf{b}$ comes along.

**Features of the LU decomposition.** This is an idealized method of matrix decomposition, which assumes that the pivots always lie on the diagonal during the elimination, so that no row exchanges are needed. Under this assumption we can express the elimination process as $\mathbf{A} = \mathbf{LU}$, where L records all the elimination multipliers and U is an upper triangular matrix. To solve a system of linear equations we only need to solve two triangular systems in turn: first $\mathbf{Ly = b}$ and then $\mathbf{Ux = y}$. The advantage of this method is that the decomposition is computed once and can be used many times, while the constant vector $\mathbf{b}$ stays unchanged. However, the existence of the LU decomposition depends on the lucky condition that "no pivot is zero," which does not always hold in practical applications.

**Features of the PLU decomposition.** This is the most complete and robust method of matrix decomposition: it introduces a permutation matrix P to record the necessary row exchanges and obtains $\mathbf{PA = LU}$. For the system of linear equations $\mathbf{Ax = b}$, we turn to solving $\mathbf{PAx = Pb}$, and then solve $\mathbf{Ly = Pb}$ and $\mathbf{Ux = y}$ in turn. Since the permutation merely rearranges the entries of a vector, computing $\mathbf{Pb}$ is very efficient. The PLU decomposition combines the efficiency of decomposition methods with the numerical stability of row exchanges, and applies to almost every matrix.

**Comparing the three methods.** In terms of computational efficiency, for a single solve Gaussian elimination has a slight advantage because it avoids storing matrices; but when many systems with the same coefficient matrix must be solved, the advantage of the PLU decomposition is significant, because the decomposition is computed only once. In terms of numerical stability, Gaussian elimination and the PLU decomposition behave identically when they use the same pivoting strategy. In memory usage, Gaussian elimination has a slight advantage because it does not need to store the matrices L and P, but this difference is usually not decisive in modern computing environments.

**Developments in modern numerical methods.** Note that the three methods above all belong to the category of direct methods, which obtain the exact solution (within numerical precision) in finitely many steps. For large sparse systems, however, modern numerical linear algebra prefers iterative methods, such as the conjugate gradient method and GMRES. These methods need no explicit matrix decomposition; instead they approach the true solution by repeatedly improving an approximate solution, and they have better computational complexity and memory efficiency on large problems. The development of iterative methods, together with advances in preconditioning, has made them the mainstream approach to solving systems of linear equations in modern scientific computing.

:::{prf:example} Complete Solution of a System of Three Linear Equations
:label: ex-gaussian-steps
Consider the system of linear equations

$$\begin{cases}
2x_0 + x_1 + x_2 = 8 \\
-x_0 + x_1 - x_2 = -2 \\
3x_0 - x_1 - x_2 = 5
\end{cases}$$

In augmented matrix form:

$$
\begin{bmatrix}
2 & 1 & 1 & | & 8 \\
-1 & 1 & -1 & | & -2 \\
3 & -1 & -1 & | & 5
\end{bmatrix}
$$

**The forward elimination process.**

1. Choose the entry 2 in row 0, column 0 as the pivot:
   - Add $\frac{1}{2}$ times row 0 to row 1:
 
     $$\begin{bmatrix}
     2 & 1 & 1 & | & 8 \\
     0 & \frac{3}{2} & -\frac{1}{2} & | & 2 \\
     3 & -1 & -1 & | & 5
     \end{bmatrix}$$
   - Add $-\frac{3}{2}$ times row 0 to row 2:
     
     $$\begin{bmatrix}
     2 & 1 & 1 & | & 8 \\
     0 & \frac{3}{2} & -\frac{1}{2} & | & 2 \\
     0 & -\frac{5}{2} & -\frac{5}{2} & | & -7
     \end{bmatrix}$$
2. Choose the entry $\frac{3}{2}$ in row 1, column 1 as the pivot:
   - Add $\frac{5}{3}$ times row 1 to row 2:

$$\begin{bmatrix}
2 & 1 & 1 & | & 8 \\
0 & \frac{3}{2} & -\frac{1}{2} & | & 2 \\
0 & 0 & -\frac{10}{3} & | & -\frac{11}{3}
\end{bmatrix}$$

3. Normalize the pivots:

- Divide row 0 by $2$:
$$\begin{bmatrix}1 & \frac{1}{2} & \frac{1}{2} & | & 4 \\0 & \frac{3}{2} & -\frac{1}{2} & | & 2 \\0 & 0 & -\frac{10}{3} & | & -\frac{11}{3}\end{bmatrix}$$

- Divide row 1 by $\frac{3}{2}$:
$$\begin{bmatrix}1 & \frac{1}{2} & \frac{1}{2} & | & 4 \\0 & 1 & -\frac{1}{3} & | & \frac{4}{3} \\0 & 0 & -\frac{10}{3} & | & -\frac{11}{3}\end{bmatrix}$$

- Divide row 2 by $-\frac{10}{3}$:
$$\begin{bmatrix}1 & \frac{1}{2} & \frac{1}{2} & | & 4 \\0 & 1 & -\frac{1}{3} & | & \frac{4}{3} \\0 & 0 & 1 & | & \frac{11}{10}\end{bmatrix}$$

4. Eliminate above the pivots:

- Add $\frac{1}{3}$ times row 2 to row 1:
$$\begin{bmatrix}1 & \frac{1}{2} & \frac{1}{2} & | & 4 \\0 & 1 & 0 & | & \frac{4}{3}+\frac{11}{30} = \frac{51}{30} = \frac{17}{10} \\0 & 0 & 1 & | & \frac{11}{10}\end{bmatrix}$$

- Add $-\frac{1}{2}$ times row 2 to row 0, and then add $-\frac{1}{2}$ times row 1 to row 0:
$$\begin{bmatrix}1 & 0 & 0 & | & 4 - \frac{11}{20} - \frac{17}{20} = 4 - \frac{28}{20} = \frac{13}{5} \\0 & 1 & 0 & | & \frac{17}{10} \\0 & 0 & 1 & | & \frac{11}{10}\end{bmatrix}$$

The final solution: $x_0 = \dfrac{13}{5}$, $x_1 = \dfrac{17}{10}$, $x_2 = \dfrac{11}{10}$.


:::

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ──────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

# ============================================================
print_header("Verification | Hand Solution of a System of Three Linear Equations (Compare with the Example Above)")
# ============================================================

# --- Step 1: define the coefficient matrix A and the right-hand-side vector b ---
print_step(1, "Define A and b")
A = np.array([
    [ 2.0,  1.0,  1.0],
    [-1.0,  1.0, -1.0],
    [ 3.0, -1.0, -1.0]
])
b = np.array([8.0, -2.0, 5.0])
print("A =\n", A)
print("b =", b)

# --- Step 2: solve with np.linalg.solve ---
print_step(2, "Solve Ax = b with np.linalg.solve")
x = np.linalg.solve(A, b)
x_hand = np.array([13/5, 17/10, 11/10])   # hand-computed result x0=13/5, x1=17/10, x2=11/10
print("numerical solution x  =", x)
print("hand solution x_hand =", x_hand)

# --- Step 3: comparison and residual ---
print_step(3, "Compare with the hand solution and check the residual")
compare_print("Maximum difference ‖x − x_hand‖∞", f"{np.max(np.abs(x - x_hand)):.2e}", "should be close to machine precision")
compare_print("Residual ‖Ax − b‖∞", f"{np.max(np.abs(A @ x_hand - b)):.2e}", "should be 0 when the hand solution is substituted back into the original system")
assert np.allclose(x, x_hand)
print("\n  ✓ The hand-computed result x = (13/5, 17/10, 11/10) passes the check.")


Gaussian elimination can also be used to find the inverse of a matrix. For an $n \times n$ matrix $\mathbf{A}$, construct the augmented matrix $[\mathbf{A}|\mathbf{I}_n]$, where $\mathbf{I}_n$ is the $n\times n$ identity matrix, and then apply Gaussian elimination to this augmented matrix. Note, however, that, as mentioned in the previous subsection, modern industry rarely computes inverse matrices; most of the time it prefers to solve the system of linear equations directly with iterative methods, and when an inverse matrix is really needed, the PLU decomposition is usually used, since it keeps the overall error under better control. Computing the inverse by Gaussian elimination in this subsection is therefore aimed more at understanding than at practical application.

:::{prf:theorem} Inversion via the Augmented Matrix
:label: thm-inverse
If the augmented matrix $[\mathbf{A}|\mathbf{I}_n]$ can be transformed by row operations into the reduced row echelon form (RREF) $[\mathbf{I}_n|\mathbf{B}]$, then $\mathbf{B} = \mathbf{A}^{-1}$. If this form cannot be obtained, then $\mathbf{A}$ is not invertible.
:::

:::{prf:proof} Proof of the Augmented-Matrix Inversion Method
:label: prf-inverse-method

1. Each elementary row operation can be expressed as left multiplication by an elementary matrix $\mathbf{E}_i$.

2. Suppose $[\mathbf{A}|\mathbf{I}_n]$ is transformed into $[\mathbf{I}_n|\mathbf{B}]$ by a sequence of row operations; this process can be expressed as
   $\mathbf{E}_k \mathbf{E}_{k-1} \cdots \mathbf{E}_0 [\mathbf{A}|\mathbf{I}_n] = [\mathbf{I}_n|\mathbf{B}]$

3. Let $\mathbf{P} = \mathbf{E}_k \mathbf{E}_{k-1} \cdots \mathbf{E}_0$; then
   $\mathbf{P}[\mathbf{A}|\mathbf{I}_n] = [\mathbf{P}\mathbf{A}|\mathbf{P}]$

4. Since $\mathbf{P}\mathbf{A} = \mathbf{I}_n$, this means that $\mathbf{P} = \mathbf{A}^{-1}$.

5. At the same time, $[\mathbf{P}\mathbf{A}|\mathbf{P}] = [\mathbf{I}_n|\mathbf{B}]$ means that $\mathbf{B} = \mathbf{P} = \mathbf{A}^{-1}$.

6. $\underline{\text{Hence, when row operations transform } [\mathbf{A}|\mathbf{I}_n] \text{ into } [\mathbf{I}_n|\mathbf{B}] \text{, the matrix } \mathbf{B} \text{ is the inverse of } \mathbf{A}}$

7. If $[\mathbf{A}|\mathbf{I}_n]$ cannot be transformed into the form $[\mathbf{I}_n|\mathbf{B}]$, that is, if the left side cannot be reduced to the identity matrix, then $\mathbf{A}$ is singular (not invertible). $\square$
:::

:::{prf:example} Computing the Inverse of a 2×2 Matrix
:label: ex-inverse
Find the inverse of the matrix $\mathbf{A} = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}$.

Construct the augmented matrix:

$$
\begin{bmatrix}
1 & 2 & | & 1 & 0 \\
3 & 4 & | & 0 & 1
\end{bmatrix}
$$

Apply Gauss–Jordan elimination:
1. Add $-3$ times row 0 to row 1:

$$
\begin{bmatrix}
1 & 2 & | & 1 & 0 \\
0 & -2 & | & -3 & 1
\end{bmatrix}
$$

2. Multiply row 1 by $-\frac{1}{2}$:

$$
\begin{bmatrix}
1 & 2 & | & 1 & 0 \\
0 & 1 & | & \frac{3}{2} & -\frac{1}{2}
\end{bmatrix}
$$

3. Add $-2$ times row 1 to row 0:

$$
\begin{bmatrix}
1 & 0 & | & -2 & 1 \\
0 & 1 & | & \frac{3}{2} & -\frac{1}{2}
\end{bmatrix}
$$

Therefore $\mathbf{A}^{-1} = \begin{bmatrix} -2 & 1 \\ \frac{3}{2} & -\frac{1}{2} \end{bmatrix}$.

One can verify that $\mathbf{A}\mathbf{A}^{-1} = \mathbf{I}_2$:

$$\begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix} \begin{bmatrix} -2 & 1 \\ \frac{3}{2} & -\frac{1}{2} \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$
:::

:::{admonition} §6.3 Summary
:class: tip

This section answered the algorithmic question "how can elimination be carried out efficiently and stably?" Algebraically, Gaussian elimination is a composition of invertible row operations, and the process can be recorded in two ways: the LU decomposition enters the elimination multipliers directly into a lower triangular matrix $\mathbf{L}$, which is elegant in form but requires the pivots never to be zero; the PLU decomposition introduces a permutation matrix $\mathbf{P}$ to record the row exchanges, which guarantees applicability to every matrix, and the resulting $\mathbf{U}$ is itself already in row echelon form. Together the two decompositions reveal a deep design principle: decompose once, use many times. When the same coefficient matrix $\mathbf{A}$ meets different right-hand-side vectors $\mathbf{b}$, we only need to solve two triangular systems for each (first $\mathbf{Ly} = \mathbf{Pb}$, then $\mathbf{Ux} = \mathbf{y}$), which greatly lowers the computational cost. The LU decomposition also contains a beautiful structure: the elimination process is equivalent to decomposing $\mathbf{A}$ into a sum of rank-one matrices, with each round of elimination contributing exactly one rank-one term. §6.4 will show the direct application of these tools in three very different fields, demonstrating the breadth and power of $\mathbf{Ax} = \mathbf{b}$ as a modeling language.
:::

## 6.4 Practical Applications of Systems of Linear Equations

Systems of linear equations have a wide range of applications in the real world; they appear everywhere from engineering design to economic analysis, and from network flows to chemical equilibrium.

### 6.4.1 Matrix Representation of Network Flow Problems

Network flow problems show the important value of systems of linear equations in practical applications. We will see that flow conservation in a network can be expressed as a **system of linear equations $\mathbf{Af} = \mathbf{b}$**, and that, because of the special properties of the incidence matrix, this system usually has infinitely many solutions. **The essence of a network flow problem is optimization over the degrees of freedom in the space formed by these solutions (in particular, the kernel of the homogeneous system $\mathbf{Af} = \mathbf{0}$)**, searching for the optimal distribution of flow subject to the capacity constraints.

:::{prf:definition} Basic Elements of a Network Flow
:label: def-network-flow-elements
Consider a directed graph $G = (V, E)$, where $V$ is the set of nodes and $E$ is the set of edges. Define:

- **Flow variables**: $f_{ij}$ denotes the flow from node $i$ to node $j$
- **Flow demands**: the flow demand $b_i$ of node $i$, where:
  - source $s$: $b_s = -$ (total supply) (a negative value)
  - sink $t$: $b_t = $ (total demand) (a positive value)  
  - intermediate nodes: $b_i = 0$ (flow conservation)
- **Capacity limits**: the maximum capacity $c_{ij}$ of each edge $(i,j)$
:::

#### Matrix Representation of the Flow Conservation Equations

:::{prf:definition} Flow Conservation Equations
:label: def-flow-conservation
For each node $i$ in the network, flow conservation requires

$$\sum_{j:(j,i) \in E} f_{ji} - \sum_{j:(i,j) \in E} f_{ij} = b_i$$

**Intuition.** The total flow into a node minus the total flow out of the node equals the flow demand of that node.

This can be written in matrix form:
$$\mathbf{Af} = \mathbf{b}$$

where:
- $\mathbf{A} \in \mathbb{R}^{|V| \times |E|}$ is the **incidence matrix**
- $\mathbf{f} \in \mathbb{R}^{|E|}$ is the **flow vector**  
- $\mathbf{b} \in \mathbb{R}^{|V|}$ is the **demand vector**
:::

:::{prf:definition} Construction of the Incidence Matrix
:label: def-incidence-matrix
Let the directed graph $G$ have $n$ nodes and $m$ edges. The entries of the incidence matrix $\mathbf{A}_{n \times m}$ are defined by

$$a_{ik} = \begin{cases}
+1 & \text{if edge } k \text{ ends at node } i \\
-1 & \text{if edge } k \text{ starts at node } i \\
0 & \text{if edge } k \text{ is not incident to node } i
\end{cases}$$

**Physical interpretation.** A positive value means inflow, and a negative value means outflow.
:::

:::{prf:example} Matrix Representation of a Simple Network
:label: ex-simple-network-matrix
Consider a network with 3 nodes and 3 edges:
- Nodes: $\{0, 1, 2\}$ (source 0, intermediate node 1, sink 2)
- Edges: $(0,1)$, $(0,2)$, $(1,2)$

The incidence matrix is

$$\mathbf{A} = \begin{bmatrix}
-1 & -1 & 0 \\
1 & 0 & -1 \\
0 & 1 & 1
\end{bmatrix}$$

The flow conservation equations:

$$\begin{bmatrix}
-1 & -1 & 0 \\
1 & 0 & -1 \\
0 & 1 & 1
\end{bmatrix}
\begin{bmatrix} f_{01} \\ f_{02} \\ f_{12} \end{bmatrix} =
\begin{bmatrix} -F \\ 0 \\ F \end{bmatrix}$$

where $F$ is the total flow from the source to the sink. Following the convention of {prf:ref}`def-incidence-matrix`, the source component $b_0 = -F$ is negative and the sink component $b_2 = F$ is positive. Checking row 0 by hand confirms this: $(\mathbf{Af})_0 = -f_{01} - f_{02} = -F$.
:::

#### Capacity Constraints

:::{prf:definition} Flow Capacity Constraints
:label: def-capacity-constraints
The flow on each edge must satisfy the capacity limit:

$$\mathbf{0} \leq \mathbf{f} \leq \mathbf{c}$$

where $\mathbf{c} = [c_{ij}]$ is the capacity vector.

**Extended form.** If flow in both directions is allowed, one can set $\mathbf{l} \leq \mathbf{f} \leq \mathbf{c}$

where $\mathbf{l}$ is the lower-bound vector (usually the zero vector, or allowing flow in the negative direction).
:::

```{important} Linear Inequality Constraints
:label: imp-linear-inequality-constraints
The capacity constraints extend the original system of linear equations into a **system of linear inequalities**, which takes the problem out of pure linear algebra and into the realm of **linear programming**.

Solving such problems requires specialized optimization algorithms, such as the simplex method or the interior point method.
```

#### The Minimum-Cost Flow Problem

:::{prf:definition} Minimum-Cost Flow
:label: def-min-cost-flow
If each edge has a cost per unit of flow $p_{ij}$, the objective function of the minimum-cost flow problem is

$$\min \mathbf{p}^{\top}\mathbf{f} = \min \sum_{(i,j) \in E} p_{ij} f_{ij}$$

**The complete optimization problem.**
$$\begin{align}
\min \quad & \mathbf{p}^{\top}\mathbf{f} \\
\text{s.t.} \quad & \mathbf{Af} = \mathbf{b} \quad \text{(flow conservation)} \\
& \mathbf{l} \leq \mathbf{f} \leq \mathbf{c} \quad \text{(capacity constraints)}
\end{align}$$

This is a standard **linear programming problem**.
:::

```{note} The Special Structure of Network Flow Problems
:label: note-network-flow-structure
Network flow problems have a special mathematical structure:

1. **Properties of the incidence matrix**: each column has exactly one $+1$ and one $-1$, and all other entries are $0$
2. **Total unimodularity**: when the supplies, demands, and upper and lower bounds are integers and a finite optimal value exists, an integer optimal solution can be chosen; it is not guaranteed that every optimal solution is an integer solution
3. **Duality**: the minimum-cost flow problem and the shortest path problem are related by duality

These properties give network flow problems efficient specialized algorithms.
```

#### Common Network Flow Algorithms

:::{tip} Main Solution Methods
:label: tip-flow-algorithms
**The maximum flow problem.**
- The Ford-Fulkerson algorithm: based on the concept of augmenting paths
- The Edmonds-Karp algorithm: uses BFS to find shortest augmenting paths
- The Push-Relabel algorithm: based on the idea of preflow-push

**The minimum-cost flow problem.**
- The successive shortest path algorithm (Successive Shortest Path)
- The cycle-canceling algorithm (Cycle-Canceling)
- The network simplex method (Network Simplex)

**The linear algebra viewpoint.** All of these algorithms are in essence solving constrained linear systems, but they exploit the special properties of the network structure to gain efficiency.
:::

:::{note} Practical Applications
:label: note-network-flow-applications
**Transportation.**
- Distribution of vehicle traffic on highway networks
- Optimization of passenger flow in airline networks
- Route planning for logistics and delivery

**Resource allocation.**
- Power flow analysis in electric power systems
- Scheduling of materials in supply chains  
- Bandwidth allocation in communication networks

**Financial engineering.**
- Risk management of cash flows
- Liquidity constraints in investment portfolios
- Analysis of interbank lending networks
:::

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# NYU color palette
NYU_TEAL, NYU_ORANGE, NYU_LIGHT_VIOLET = '#2AD2C9', '#FF5D47', '#AB82C5'
# Create the list of nodes, including the source 's' and the sink 't'
nodes = ['s', 'a', 'b', 'c', 't']

# Define the incidence matrix to represent the flow conservation equations
# rows: nodes ['s', 'a', 'b', 'c', 't']
# columns: edges [('s','a'), ('s','b'), ('a','b'), ('a','c'), ('b','c'), ('b','t'), ('c','t')]
# values: +1 means inflow (the node is the end of the edge), -1 means outflow (the node is the start of the edge), 0 means not connected
# —— exactly the same convention as def-incidence-matrix in the text
incidence_matrix = np.array([
    [-1, -1,  0,  0,  0,  0,  0],  # s: outflow on edges (s,a) and (s,b)
    [ 1,  0, -1, -1,  0,  0,  0],  # a: inflow (s,a), outflow (a,b) and (a,c)
    [ 0,  1,  1,  0, -1, -1,  0],  # b: inflow (s,b) and (a,b), outflow (b,c) and (b,t)
    [ 0,  0,  0,  1,  1,  0, -1],  # c: inflow (a,c) and (b,c), outflow (c,t)
    [ 0,  0,  0,  0,  0,  1,  1]   # t: inflow (b,t) and (c,t)
])

# Define the edges and their capacities
edges = [('s', 'a', 10), ('s', 'b', 8),
         ('a', 'b', 5), ('a', 'c', 4),
         ('b', 'c', 6), ('b', 't', 7),
         ('c', 't', 10)]

# Create the directed graph from the matrix
G = nx.DiGraph()
G.add_nodes_from(nodes)

for u, v, capacity in edges:
    G.add_edge(u, v, capacity=capacity)

# Set the node positions for visualization
pos = {'s': (0, 1), 'a': (1, 2), 'b': (1, 0), 'c': (2, 1), 't': (3, 1)}

# Compute the maximum flow
max_flow_value, flow_dict = nx.maximum_flow(G, 's', 't')

# Set the edge labels in the format "flow/capacity"
edge_labels = {}
for u, v, capacity in edges:
    flow = flow_dict[u][v]
    edge_labels[(u, v)] = f"{flow}/{capacity}"

# Give the source and the sink special colors (NYU color palette)
node_colors = []
for node in nodes:
    if node == 's':
        node_colors.append(NYU_TEAL)          # source: Mercer Teal
    elif node == 't':
        node_colors.append(NYU_ORANGE)        # sink: Hudson Orange
    else:
        node_colors.append(NYU_LIGHT_VIOLET)  # intermediate nodes: Mayfair Light Violet

# Draw the network flow graph
plt.figure(figsize=(10, 6))
nx.draw(G, pos, with_labels=True, node_color=node_colors, 
        node_size=500, font_weight='bold', arrows=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.title("Network Flow Graph Built from the Flow Conservation Matrix")
plt.axis('off')
plt.tight_layout()
plt.show()

# Print the incidence matrix
print("Incidence matrix of flow conservation:")
print(incidence_matrix)

# Print the flow conservation equation of each node
print("\nFlow conservation equation of each node:")
edge_names = [f"f({u},{v})" for u, v, _ in edges]

for i, node in enumerate(nodes):
    eq_parts = []
    for j, (u, v, _) in enumerate(edges):
        if incidence_matrix[i, j] == 1:
            eq_parts.append(f"+{edge_names[j]}")
        elif incidence_matrix[i, j] == -1:
            eq_parts.append(f"-{edge_names[j]}")
    
    equation = "".join(eq_parts)
    if node == 's':
        print(f"Source '{node}': {equation} = -v (v is the total flow; by convention the net flow at the source is negative)")
    elif node == 't':
        print(f"Sink '{node}': {equation} = v  (v is the total flow; by convention the net flow at the sink is positive)")
    else:
        print(f"Node '{node}': {equation} = 0  (inflow = outflow)")

# The maximum flow was already computed in the plotting part
print(f"\nMaximum flow value: {max_flow_value}")

# Show the flow on each edge
print("\nOptimal flow assignment:")
flow_vector = []
for u, v, _ in edges:
    flow = flow_dict[u][v]
    flow_vector.append(flow)
    print(f"f({u},{v}) = {flow}")

# Verify flow conservation: check A·f = b node by node
flow_vector = np.array(flow_vector)
node_net_flows = incidence_matrix @ flow_vector

# Construct the demand vector b by the def-incidence-matrix convention: source -v, sink +v, intermediate nodes 0
b = np.zeros(len(nodes))
b[nodes.index('s')] = -max_flow_value
b[nodes.index('t')] =  max_flow_value

print("\nVerify flow conservation (A·f = b):")
for i, node in enumerate(nodes):
    if node == 's':
        print(f"Source '{node}': net flow = {node_net_flows[i]} (should be -{max_flow_value})")
    elif node == 't':
        print(f"Sink '{node}': net flow = {node_net_flows[i]} (should be {max_flow_value})")
    else:
        print(f"Node '{node}': net flow = {node_net_flows[i]} (should be 0)")

assert np.allclose(incidence_matrix @ flow_vector, b), "A·f should equal b"
print("\n  ✓ A·f = b checked node by node: the code, the definition, and the example use the same sign convention (+1 = inflow).")

### 6.4.2 A Case Study in Balancing Chemical Equations

Balancing a chemical reaction equation is in essence solving a system of linear equations.

:::{prf:example} Balancing the Combustion of an Alkane
:label: ex-chemical-balance
Consider the unbalanced chemical equation
$C_3H_8 + O_2 \to CO_2 + H_2O$

Let the coefficients of the substances be $a$, $b$, $c$, and $d$, respectively.

Writing the equations for the conservation of each element:
- C: $3a = c$
- H: $8a = 2d$
- O: $2b = 2c + d$

This forms a system of linear equations. Taking $a=1$ as the reference, we obtain
$c = 3$
$d = 4$
$b = 5$

Hence the balanced equation is
$C_3H_8 + 5O_2 \to 3CO_2 + 4H_2O$
:::

```{note}
:label: note-chemical-balance
The problem of balancing a chemical equation usually has infinitely many solutions (scaling all the coefficients by the same factor also gives a solution), but we usually choose the solution whose coefficients are the smallest positive integers.
```

### 6.4.3 Mechanical Equilibrium

In mechanics problems, a body acted on by several forces remains in equilibrium, and this can be described by a system of linear equations.

:::{prf:example} Mechanical Equilibrium of a Weight Suspended by Two Ropes
:label: ex-mechanical-equilibrium
Consider a weight suspended from two ropes. Let the tensions in the two ropes be $T_0$ and $T_1$, and let the weight of the object be $W$.

By the equilibrium condition (the vector sum of the forces is zero), we can write:
- Horizontal direction: $T_0\cos\theta_0 - T_1\cos\theta_1 = 0$
- Vertical direction: $T_0\sin\theta_0 + T_1\sin\theta_1 - W = 0$

This is a system of linear equations in $T_0$ and $T_1$.

Suppose $\theta_0 = 30^\circ$, $\theta_1 = 45^\circ$, $W = 100$ N; then
- $T_0 \cdot 0.866 - T_1 \cdot 0.707 = 0$
- $T_0 \cdot 0.5 + T_1 \cdot 0.707 = 100$

**Solving by elimination** (even for a $2 \times 2$ system, we go through the procedure of this chapter): add equation 0 to equation 1 to eliminate $T_1$:

$$(0.5 + 0.866)\,T_0 = 100 \quad\Rightarrow\quad 1.366\,T_0 = 100 \quad\Rightarrow\quad T_0 \approx 73.2 \text{ N}$$

Back-substituting into equation 0: $T_1 = \dfrac{0.866}{0.707} \times 73.2 \approx 89.7$ N.

**Check** (substitute the solution back into the two original equations):
- Horizontal: $0.866 \times 73.2 - 0.707 \times 89.7 \approx 63.4 - 63.4 = 0$ ✓
- Vertical: $0.5 \times 73.2 + 0.707 \times 89.7 \approx 36.6 + 63.4 = 100$ ✓

Solution: $T_0 \approx 73.2$ N, $T_1 \approx 89.7$ N.
:::

```{important}
:label: imp-applications
The power of systems of linear equations as a modeling tool lies in their ability to turn complex practical problems into mathematical problems that can be solved systematically by algebraic methods. Linear algebra provides a unified theoretical framework and effective algorithms, so that problems that look different on the surface can be handled in the same way.
```

**Systems of linear equations have wide applications in every field.** By turning network flow problems, chemical equilibrium, and mechanical equilibrium into systems of linear equations, we can analyze and solve these problems systematically. The theoretical framework and computational tools provided by linear algebra make complex problems solvable, showing the power of mathematics in practical applications.

:::{admonition} §6.4 Summary
:class: tip

This section answered the applied question "how far can the theory of systems of linear equations reach?" Flow conservation at the nodes of a network, the conservation of atoms in chemical reactions, and the static equilibrium of mechanical systems—these three problems from engineering, chemistry, and physics are all captured by the single unified language of $\mathbf{Ax} = \mathbf{b}$. The special structure of the incidence matrix (each column contains only $\pm 1$ as nonzero entries) makes the network flow equations usually underdetermined, and the degrees of freedom of the solution space correspond to the number of loops in the network; the systems of equations for chemical balancing are also often underdetermined, and by convention the solution in smallest positive integers is chosen; the two tensions in our example have a unique solution when $\sin(\theta_0+\theta_1)\neq0$ (for example, when $0<\theta_0,\theta_1<90^\circ$). The three applications share one body of theory: the analysis of the structure of solutions in §6.2 determines "how many degrees of freedom there are," and the PLU decomposition of §6.3 provides the practical algorithm for solving. This unity across fields is the fundamental reason why linear algebra has become a common language of modern science, and it also lays the groundwork for Chapter 8, which studies when the homogeneous equation with a parameter $(\mathbf{A}-\lambda\mathbf{I})\mathbf{v}=\mathbf{0}$ has a nonzero solution, and for Chapter 11, which treats "overdetermined or underdetermined" least-squares problems (SVD).
:::

## 6.5 Chapter Summary

### Review of the Theoretical Thread

This chapter started from the premise that "row operations are invertible linear transformations" and built a complete theoretical framework for solving systems of linear equations. §6.1 laid the foundation for the legitimacy of the operations: the three elementary row operations (row scaling, row exchange, row elimination) are each equivalent to left multiplication by an elementary matrix, and so they are invertible, change neither the solution set nor the rank of the matrix. The theorem on the preservation of subspace dimensions gives the most general statement—left or right multiplication by any invertible matrix preserves the dimensions of the row space and the column space—which guarantees the legitimacy of every subsequent reduction step.

§6.2 used this guarantee and, centered on the block structure $\begin{bmatrix}\mathbf{I}_r & \mathbf{F} \\ \mathbf{0} & \mathbf{0}\end{bmatrix}$ of the RREF, gave a complete description of the geometric form of the solutions. The rank $r$ determines all the dimensions of the four subspaces; the number of free variables, $n - r$, determines the dimension of the solution space; and the relation $\mathbf{y}_p = -\mathbf{F}\mathbf{y}_f$ between the pivot variables and the free variables directly gives a parametric representation of the solution space. The three forms—no solution, a unique solution, infinitely many solutions—become clear at a glance from this block viewpoint, and the abstract question of "solvability" is reduced to a computable "comparison of ranks."

§6.3 provided the algorithmic tools for bringing a general matrix to RREF. The LU decomposition is elegant in form and records the elimination multipliers directly as the lower triangular matrix $\mathbf{L}$, but it depends on the precondition that no pivot is zero; the PLU decomposition records the row exchanges in a permutation matrix $\mathbf{P}$, applies to every matrix, and the resulting $\mathbf{U}$ is itself already in row echelon form. The two share the same design philosophy: decompose once, use many times, so that when the same coefficient matrix meets different right-hand-side vectors, only two triangular systems need to be solved for each. §6.4 then used three very different applications—network flow, chemical balancing, and mechanical equilibrium—to show the breadth of $\mathbf{Ax} = \mathbf{b}$ as a unified modeling language.

### Connections to Other Chapters

The link between this chapter and Chapter 5 (block matrices) lies in the block structure of the RREF itself: Chapter 5 established the algebraic rules for operating with block matrices, and this chapter applies them to reveal the rank structure of a matrix and the geometric form of the solution space. The rigorous derivation of the PLU decomposition relies on the multiplicative properties of block matrices, in particular on the key property that "the commutation relation between permutation matrices and lower triangular matrices allows the row exchanges to be moved, equivalently, to the very front of the decomposition"—a conclusion that becomes natural only from the block viewpoint.

The connection with Chapter 7 (determinants) is even more direct: the basic way of computing a determinant is precisely to bring the matrix to upper triangular form by row operations, and the effects of the row operations on the determinant—scaling multiplies it by $c$, an exchange changes its sign, elimination leaves its value unchanged—all follow naturally from the elementary-matrix framework of this chapter. The concept of rank in this chapter also finally merges with the theory of dimension of abstract linear spaces in Chapter 4: the concrete computational statement that "the number of pivots equals the dimension of the image" is exactly the dimension theorem of Chapter 4 expressed in the language of coordinates. Looking further ahead, the idea of the LU decomposition will reappear in Chapter 8 (the eigenvalue problem) and Chapter 11 (the singular value decomposition) in the forms of the Schur decomposition, the QR decomposition, and the SVD; "factoring a matrix into a product of factors of simpler structure" is a central theme that runs throughout the book.

### The Role of This Chapter in the Book

Systems of linear equations are the oldest problem in linear algebra and also its most central one. The significance of this chapter lies not in supplying techniques for "solving equations," but in revealing the answers to two essential questions: "what determines solvability?" and "what is the structure of the solutions?"; and in raising the arithmetic elimination already recorded in *The Nine Chapters on the Mathematical Art* to a rigorous algebraic language. Elimination operations are isomorphisms, the RREF is a canonical form, and the solution set is an affine subspace—these three sentences capture the full depth of this chapter. If you keep reading with this viewpoint, you will find that every later chapter answers the same deeper question: how to find a "canonical form" in which the essential structure of a matrix is laid bare. This chapter is the first complete demonstration of this theme.

### Concept Map
```{mermaid}
%%{init: {
  "theme": "base",
  "themeVariables": {
    "background":         "#FFFFFF",
    "primaryColor":       "#FFFFFF",
    "primaryTextColor":   "#57068C",
    "primaryBorderColor": "#57068C",
    "lineColor":          "#7F8C8D",
    "secondaryColor":     "#F1F2F6",
    "tertiaryColor":      "#FFFFFF",
    "clusterBkg":         "#F9F9FB",
    "clusterBorder":      "#DCDDE1",
    "fontSize":           "14px"
  }
}}%%
graph TD

  subgraph S0 ["<span style='color:#7F8C8D'>Prerequisite Chapters</span>"]
    PREV["Chapter 5<br/>Block Matrices"]
    PREV4["Chapter 4<br/>Dimension Theorem"]
  end

  subgraph S1 ["<span style='color:#57068C;font-weight:bold'>§6.1 Elementary Operations</span>"]
    N01["Elementary Matrices<br/>Row/Column Operations"]
    N02["Rank-Preservation Theorem<br/>Invertible Transformations"]
  end

  subgraph S2 ["<span style='color:#006385;font-weight:bold'>§6.2 RREF and the Structure of Solutions</span>"]
    N03["RREF Blocks<br/>I_r F / 0 0"]
    N04["Dimensions of the<br/>Four Subspaces"]
    N05["Three Forms of Solutions<br/>Unique/None/Infinitely Many"]
  end

  subgraph S3 ["<span style='color:#087D77;font-weight:bold'>§6.3 Elimination</span>"]
    N06["LU Decomposition<br/>A = LU"]
    N07["PLU Decomposition<br/>PA = LU"]
    N08["Unified Solution Framework<br/>Efficient Solution of Many Systems"]
  end

  subgraph S4 ["<span style='color:#2F3640'>§6.4 Applications</span>"]
    N09["Network Flow<br/>Af = b"]
    N10["Chemical Balancing<br/>Conservation of Atoms"]
    N11["Mechanical Equilibrium"]
  end

  subgraph S5 ["<span style='color:#7F8C8D'>Later Chapters</span>"]
    NEXT7["Chapter 7<br/>Determinants"]
    NEXT8["Chapter 8<br/>Eigenvalues"]
    NEXT11["Chapter 11<br/>SVD"]
  end

  PREV  -.->|builds on| N01
  PREV4 -.->|builds on| N02
  N01 --> N02
  N02 --> N03
  N03 --> N04
  N03 --> N05
  N04 --> N06
  N05 --> N06
  N06 --> N07
  N07 --> N08
  N07 --> N09
  N07 --> N10
  N07 --> N11
  N02 -.->|prepares for| NEXT7
  N07 -.->|prepares for| NEXT8
  N07 -.->|prepares for| NEXT11

  classDef main      fill:#F4EEF9,stroke:#57068C,stroke-width:2px,color:#57068C
  classDef structure fill:#E6F0F3,stroke:#006385,stroke-width:2px,color:#006385
  classDef accent    fill:#EAF9F8,stroke:#2AD2C9,stroke-width:2px,color:#087D77,font-weight:bold
  classDef neutral   fill:#FFFFFF,stroke:#DCDDE1,color:#2F3640
  classDef future    fill:#FFFFFF,stroke:#AB82C5,stroke-dasharray:5 5,color:#7F8C8D

  class N01,N02 main
  class N03,N04,N05 structure
  class N06,N07,N08 accent
  class N09,N10,N11 neutral
  class PREV,PREV4,NEXT7,NEXT8,NEXT11 future
```